<a href="https://colab.research.google.com/github/Takumi173/Test/blob/main/Dataset_JSON_Reviewer_JSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 準備

## 処理用にデータを結合

In [25]:
# データのコピー
!git clone https://github.com/cdisc-org/sdtm-adam-pilot-project.git

fatal: destination path 'sdtm-adam-pilot-project' already exists and is not an empty directory.


In [26]:
# 使用するjsonデータとdefine.xmlを新規フォルダにコピーする

import os
import shutil
import json

source_dir = "sdtm-adam-pilot-project/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm"
json_dir   = "json_files"
define_dir = "define_xml"

if not os.path.exists(json_dir):
    os.makedirs(json_dir)

if not os.path.exists(define_dir):
    os.makedirs(define_dir)

for root, _, files in os.walk(source_dir):
  for file in files:
    if file.endswith(".json"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(json_dir, file)
      shutil.copy(source_path, target_path)
    if file.endswith("define.xml"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(define_dir, file)
      shutil.copy(source_path, target_path)

In [27]:
# jsonファイルをリスト形式に結合したファイル（dataset_list.json）を作成

dataset_list = []
for filename in os.listdir(json_dir):
  if filename.endswith(".json"):
    with open(os.path.join(json_dir, filename), "r") as f:
      try:
        json_data = json.load(f)
        dataset_list.append(json_data)
      except json.JSONDecodeError as e:
        print(f"Error decoding JSON in file {filename}: {e}")

with open("dataset_list.json", "w") as f:
  json.dump(dataset_list, f)


## 症例フィルタリング関数の定義

In [28]:
def filter_data(data, target_usubjids):
    """
    複数のドメインデータを含むリストから、指定されたUSUBJIDのrowsのみを抽出して新しいJSONファイルに保存する。
    入力データがリストでない場合はエラーメッセージを出力する。
    データ構造は、"columns" 内の "name" が "USUBJID" の列を持つことを前提とする。

    Args:
        data (list): ドメインを結合させたのリスト。リストでない場合はエラーとなる。
        output_file (str): 出力するJSONファイル名。
        target_usubjids (list): 残したいUSUBJIDのリスト。
    """
    if not isinstance(data, list):
        print("エラー：入力データはJSONオブジェクトのリストである必要があります。")
        return

    filtered_data_list = []
    for item in data:
        usubjid_index = -1
        if 'columns' in item:
            for i, col in enumerate(item['columns']):
                if 'name' in col and col['name'] == 'USUBJID':
                    usubjid_index = i
                    break

        if usubjid_index == -1:
            print(f"警告：データセット '{item.get('fileOID', '不明')}' に 'name' が 'USUBJID' の列が見つかりません。スキップします。")
            filtered_data_list.append(item)
            continue

        if 'rows' in item:
            filtered_rows = [
                row for row in item['rows'] if len(row) > usubjid_index and row[usubjid_index] in target_usubjids
            ]
            new_data = item.copy()
            new_data['rows'] = filtered_rows
            new_data['records'] = len(filtered_rows)
            filtered_data_list.append(new_data)
        else:
            print(f"警告：データセット '{item.get('fileOID', '不明')}' に 'rows' が見つかりません。スキップします。")
            filtered_data_list.append(item)

    return filtered_data_list



# 実行テスト
# with open('dataset_list.json', 'r') as f:
#   data = json.load(f)
#
# target_ids = ['01-701-1211']
# output_filename = 'filtered_list.json'
#
# filtered_data_list = filter_data(data, target_ids)
#
# with open(output_filename, 'w') as f:
#   json.dump(filtered_data_list, f)
#
# print(f"処理完了：'{output_filename}' に USUBJID が {target_ids} のデータを出力しました。")

## データ書き換え関数の定義

In [29]:
def data_update(data, target_domain, target_usubjid, target_seq, target_variable, new_value):
    """
    指定されたUSUBJIDを持つレコードの指定された変数を書き換えます。
    {target_domain}SEQが存在する場合はそれもキーとして使用します。
    元のデータは変更せず、新しいデータ構造を返します。

    Args:
        data (list): データ全体のリスト。指定されたtarget_domainのデータセットを含むことを想定します。
        target_domain (str): 対象のドメイン名（例: "CM"）。
        target_usubjid (str): 書き換えたいレコードのUSUBJID。
        target_seq (int): 書き換えたいレコードの{target_domain}SEQの値（存在しない場合は無視されます）。
        target_variable (str): 書き換えたい変数の名前（例: "CMTRT"）。
        new_value (any): 新しい変数の値。

    Returns:
        list: 指定された変数が更新された新しいデータ全体のリスト。
              該当するレコードが見つからなかった場合、元のデータのコピーを返します。
    """
    updated_data = []
    seqname = target_domain + 'SEQ'

    for dataset in data:
        updated_dataset = dataset.copy()
        if updated_dataset.get("itemGroupOID") == target_domain:
            updated_rows = []
            found = False
            usubjid_index = -1
            seq_index = -1
            variable_index = -1
            has_seq = False

            for i, col in enumerate(updated_dataset["columns"]):
                if col["name"] == "USUBJID":
                    usubjid_index = i
                elif col["name"] == seqname:
                    seq_index = i
                    has_seq = True
                elif col["name"] == target_variable:
                    variable_index = i

            if usubjid_index != -1 and variable_index != -1:
                for row in dataset["rows"]:
                    updated_row = list(row)  # 行をコピーして変更
                    usubjid_match = updated_row[usubjid_index] == target_usubjid
                    seq_match = True
                    if has_seq and seq_index != -1:
                        seq_match = (len(updated_row) > seq_index and updated_row[seq_index] == target_seq)
                    elif has_seq:
                        print(f"警告: '{target_domain}' データセットに '{seqname}' 列が見つかりましたが、インデックスが無効です。USUBJIDのみをキーとして使用します。")

                    if usubjid_match and seq_match:
                        updated_row[variable_index] = new_value
                        if has_seq and seq_index != -1:
                            print(f"USUBJID '{target_usubjid}'、'{seqname}' '{target_seq}' の '{target_variable}' を '{new_value}' に更新しました。")
                        else:
                            print(f"USUBJID '{target_usubjid}' の '{target_variable}' を '{new_value}' に更新しました。")
                        found = True
                    updated_rows.append(updated_row)
                updated_dataset["rows"] = updated_rows
            elif updated_dataset.get("itemGroupOID") == target_domain:
                print(f"'{target_domain}' データセットに 'USUBJID' または '{target_variable}' 列が見つかりませんでした。")

            updated_data.append(updated_dataset)
            if not found and updated_dataset.get("itemGroupOID") == target_domain:
                if has_seq and seq_index != -1:
                    print(f"USUBJID '{target_usubjid}'、'{seqname}' '{target_seq}' に該当するレコードが見つかりませんでした。")
                else:
                    print(f"USUBJID '{target_usubjid}' に該当するレコードが見つかりませんでした。")
        else:
            updated_data.append(updated_dataset)

    if not any(d.get("itemGroupOID") == target_domain for d in data):
        print(f"{target_domain} データセットが見つかりませんでした。")

    return updated_data

# 書き換えテスト
# updated_data = data_update(filtered_data_list, "DM", "01-701-1211", 0, "AGE", 49)
# updated_data = data_update(updated_data, "CM", "01-701-1211", 3, "CMTRT", "New Drug 123456789")
# updated_data = data_update(updated_data, "CM", "01-701-1211", 0, "CMDOSE", 123)

## データ比較関数の定義

In [30]:
from typing import List, Dict, Any

def compare_data(old_data: List[Dict[str, Any]], new_data: List[Dict[str, Any]]) -> None:
    """
    2つのデータリストの更新差分を人間が読みやすい形式で出力します。

    Args:
        old_data: 旧データリスト。
        new_data: 新データリスト。
    """

    def create_row_dict(item_group: Dict[str, Any], row: List[Any]) -> Dict[str, Any]:
        """rowデータをキー付きの辞書に変換する"""
        row_dict = {}
        for i, column in enumerate(item_group['columns']):
            row_dict[column['name']] = row[i]
        return row_dict

    def get_key_values(item_group_oid: str, row_dict: Dict[str, Any]) -> Dict[str, Any]:
        """データのキーとなる値を抽出する"""
        key_values = {'USUBJID': row_dict.get('USUBJID')}
        seq_key = f"{item_group_oid}SEQ"
        if seq_key in row_dict:
            key_values[seq_key] = row_dict[seq_key]
        return key_values

    def format_key(key_values: Dict[str, Any]) -> str:
        """キー値を人間が読みやすい文字列に整形する"""
        parts = []
        for key, value in key_values.items():
            if value is not None:
                parts.append(f"{key} = {value}")
        return ", ".join(parts)

    old_data_by_group = {item['itemGroupOID']: item for item in old_data}
    new_data_by_group = {item['itemGroupOID']: item for item in new_data}

    all_group_oids = set(old_data_by_group.keys()) | set(new_data_by_group.keys())

    for group_oid in sorted(list(all_group_oids)):
        print(f"--- ItemGroupOID: {group_oid} ---")
        old_group = old_data_by_group.get(group_oid)
        new_group = new_data_by_group.get(group_oid)

        old_rows_by_key = {}
        if old_group and 'rows' in old_group:
            for row in old_group['rows']:
                row_dict = create_row_dict(old_group, row)
                if 'USUBJID' in row_dict and row_dict['USUBJID'] is not None:
                    key_values = get_key_values(group_oid, row_dict)
                    old_rows_by_key[format_key(key_values)] = row_dict

        new_rows_by_key = {}
        if new_group and 'rows' in new_group:
            for row in new_group['rows']:
                row_dict = create_row_dict(new_group, row)
                if 'USUBJID' in row_dict and row_dict['USUBJID'] is not None:
                    key_values = get_key_values(group_oid, row_dict)
                    new_rows_by_key[format_key(key_values)] = row_dict

        old_keys = set(old_rows_by_key.keys())
        new_keys = set(new_rows_by_key.keys())

        # 追加されたデータ
        added_keys = new_keys - old_keys
        for key in sorted(list(added_keys)):
            print(f"{key}:")
            print("  Added")
            for item_key, old_value in sorted(new_rows_by_key[key].items()):
                print(f"    {item_key}: {old_value}")
            print()

        # 削除されたデータ
        removed_keys = old_keys - new_keys
        for key in sorted(list(removed_keys)):
            print(f"{key}:")
            print("  Deleted")
            for item_key, old_value in sorted(old_rows_by_key[key].items()):
                print(f"    {item_key}: {old_value}")
            print()

        # 更新されたデータ
        common_keys = old_keys & new_keys
        for key in sorted(list(common_keys)):
            if old_rows_by_key[key] != new_rows_by_key[key]:
                print(f"{key}:")
                print("  Updated:")
                old_row = old_rows_by_key[key]
                new_row = new_rows_by_key[key]
                for item_key in sorted(list(set(old_row.keys()) | set(new_row.keys()))):
                    old_value = old_row.get(item_key)
                    new_value = new_row.get(item_key)
                    if old_value != new_value:
                        print(f"    {item_key}: {old_value!r} -> {new_value!r}")
                print()


# 比較テスト
# compare_data(filtered_data_list, updated_data)

In [31]:
import json

usubjids = set()
for dataset in dataset_list:
    if 'columns' in dataset:
        for i, col in enumerate(dataset['columns']):
            if 'name' in col and col['name'] == 'USUBJID':
                if 'rows' in dataset:
                    for row in dataset['rows']:
                        if len(row) > i:
                            usubjids.add(row[i])

print(list(usubjids))


['01-704-1025', '01-701-1341', '01-705-1018', '01-709-1029', '01-710-1278', '01-716-1003', '01-718-1172', '01-701-1028', '01-709-1007', '01-701-1360', '01-703-1096', '01-707-1430', '01-707-1434', '01-710-1045', '01-715-1405', '01-716-1308', '01-718-1254', '01-716-1298', '01-709-1237', '01-701-1240', '01-703-1379', '01-701-1386', '01-711-1163', '01-718-1355', '01-708-1158', '01-716-1026', '01-716-1157', '01-709-1312', '01-708-1296', '01-709-1102', '01-701-1211', '01-701-1345', '01-703-1076', '01-704-1323', '01-711-1283', '01-715-1155', '01-710-1183', '01-717-1174', '01-716-1151', '01-710-1380', '01-718-1328', '01-701-1015', '01-716-1364', '01-704-1325', '01-715-1085', '01-715-1397', '01-716-1189', '01-704-1065', '01-718-1079', '01-707-1037', '01-708-1178', '01-708-1348', '01-703-1439', '01-701-1440', '01-704-1009', '01-710-1129', '01-710-1235', '01-710-1154', '01-703-1042', '01-708-1272', '01-709-1285', '01-710-1002', '01-710-1314', '01-704-1241', '01-702-1082', '01-703-1335', '01-710-1

# データの書き換え

In [32]:
Target_data = [
["DM", "01-703-1096",   0, "AGE", 49],
["LB", "01-703-1042",   3, "LBORRES", "135"],
["LB", "01-703-1042",   4, "LBORRES", "145"],
["LB", "01-703-1086",  37, "LBORRES", "1"],
["LB", "01-703-1086",  72, "LBORRES", "1.2"],
["LB", "01-703-1086", 102, "LBORRES", "1.1"],
["LB", "01-703-1086", 132, "LBORRES", "1"],
["LB", "01-703-1086", 162, "LBORRES", "1.3"],
["LB", "01-703-1086", 197, "LBORRES", "0.9"],
["LB", "01-703-1086", 232, "LBORRES", "0.8"],
["LB", "01-703-1042",   3, "LBSTRESC", "135"],
["LB", "01-703-1042",   4, "LBSTRESC", "145"],
["LB", "01-703-1086",  37, "LBSTRESC", "1"],
["LB", "01-703-1086",  72, "LBSTRESC", "1.2"],
["LB", "01-703-1086", 102, "LBSTRESC", "1.1"],
["LB", "01-703-1086", 132, "LBSTRESC", "1"],
["LB", "01-703-1086", 162, "LBSTRESC", "1.3"],
["LB", "01-703-1086", 197, "LBSTRESC", "0.9"],
["LB", "01-703-1086", 232, "LBSTRESC", "0.8"],
["LB", "01-703-1042",   3, "LBSTRESN", 135],
["LB", "01-703-1042",   4, "LBSTRESN", 145],
["LB", "01-703-1086",  37, "LBSTRESN", 1],
["LB", "01-703-1086",  72, "LBSTRESN", 1.2],
["LB", "01-703-1086", 102, "LBSTRESN", 1.1],
["LB", "01-703-1086", 132, "LBSTRESN", 1],
["LB", "01-703-1086", 162, "LBSTRESN", 1.3],
["LB", "01-703-1086", 197, "LBSTRESN", 0.9],
["LB", "01-703-1086", 232, "LBSTRESN", 0.8],
["LB", "01-703-1042",   3, "LBNRIND", "HIGH"],
["LB", "01-703-1042",   4, "LBNRIND", "HIGH"],
["LB", "01-703-1086",  37, "LBNRIND", "LOW"],
["LB", "01-703-1086",  72, "LBNRIND", "LOW"],
["LB", "01-703-1086", 102, "LBNRIND", "LOW"],
["LB", "01-703-1086", 132, "LBNRIND", "LOW"],
["LB", "01-703-1086", 162, "LBNRIND", "LOW"],
["LB", "01-703-1086", 197, "LBNRIND", "LOW"],
["LB", "01-703-1086", 232, "LBNRIND", "LOW"],
["MH", "01-701-1097",   1, "MHTERM", "Loss of consciousness (Passed out)"],
["MH", "01-701-1097",   1, "MHSTDTC", "2023-01-01"],
["MH", "01-701-1111",   1, "MHTERM", "HEARING LOSS"],
["MH", "01-701-1180",   1, "MHTERM", "DEPRESSION (ANXIETY)"],
["MH", "01-702-1082",   1, "MHTERM", "Premenstrual pain"],
["MH", "01-703-1076",   1, "MHTERM", "Atrioventricular block (scheduled cardiac pacemaker insertion)"],
["MH", "01-703-1279",   1, "MHTERM", "schizophreniform disorders"],
["MH", "01-703-1299",   1, "MHTERM", "Cyclothymic disorder"],
["VS", "01-701-1047",  17, "VSORRES", "121"],
["VS", "01-701-1047",  18, "VSORRES", "124"],
["VS", "01-701-1047",  66, "VSORRES", "185"],
["VS", "01-701-1047",  67, "VSORRES", "183"],
["VS", "01-701-1383",  37, "VSORRES", "98"],
["VS", "01-701-1383", 122, "VSORRES", "160"],
["VS", "01-701-1387",   1, "VSORRES", "146"],
["VS", "01-701-1387",  32, "VSORRES", "72"],
["VS", "01-701-1047",  17, "VSSTRESC", "121"],
["VS", "01-701-1047",  18, "VSSTRESC", "124"],
["VS", "01-701-1047",  66, "VSSTRESC", "185"],
["VS", "01-701-1047",  67, "VSSTRESC", "183"],
["VS", "01-701-1383",  37, "VSSTRESC", "98"],
["VS", "01-701-1383", 122, "VSSTRESC", "160"],
["VS", "01-701-1387",   1, "VSSTRESC", "146"],
["VS", "01-701-1387",  32, "VSSTRESC", "72"],
["VS", "01-701-1047",  17, "VSSTRESN", 121],
["VS", "01-701-1047",  18, "VSSTRESN", 124],
["VS", "01-701-1047",  66, "VSSTRESN", 185],
["VS", "01-701-1047",  67, "VSSTRESN", 183],
["VS", "01-701-1383",  37, "VSSTRESN", 98],
["VS", "01-701-1383", 122, "VSSTRESN", 160],
["VS", "01-701-1387",   1, "VSSTRESN", 146],
["VS", "01-701-1387",  32, "VSSTRESN", 72],
["EX", "01-701-1148",   2, "EXDOSE", 82],
["EX", "01-701-1148",   3, "EXDOSE", 216],
["EX", "01-703-1258",   2, "EXDOSE", 27],
["CM", "01-701-1146",  29, "CMTRT", "PAROXETINE"],
["QS", "01-701-1023",1010, "QSORRES", "PRESENT"],
["QS", "01-701-1023",1012, "QSORRES", "PRESENT"],
["QS", "01-701-1111",5004, "QSORRES", "4"],
["QS", "01-701-1111",5019, "QSORRES", "4"],
["QS", "01-701-1111",5012, "QSORRES", "4"],
["QS", "01-701-1111",5027, "QSORRES", "4"],
["QS", "01-701-1118",6002, "QSORRES", "MARKED IMPROVEMENT"],
["QS", "01-701-1118",6003, "QSORRES", "MARKED WORSENING"],
["QS", "01-701-1181",4018, "QSORRES", "Y"],
["QS", "01-701-1181",4058, "QSORRES", "Y"],
["QS", "01-701-1181",4019, "QSORRES", "Y"],
["QS", "01-701-1181",4059, "QSORRES", "Y"],
["QS", "01-701-1181",4020, "QSORRES", "Y"],
["QS", "01-701-1023",1010, "QSSTRESC", "2"],
["QS", "01-701-1023",1012, "QSSTRESC", "2"],
["QS", "01-701-1111",5004, "QSSTRESC", "4"],
["QS", "01-701-1111",5019, "QSSTRESC", "4"],
["QS", "01-701-1111",5012, "QSSTRESC", "4"],
["QS", "01-701-1111",5027, "QSSTRESC", "4"],
["QS", "01-701-1118",6002, "QSSTRESC", "1"],
["QS", "01-701-1118",6003, "QSSTRESC", "7"],
["QS", "01-701-1181",4018, "QSSTRESC", "1"],
["QS", "01-701-1181",4058, "QSSTRESC", "1"],
["QS", "01-701-1181",4019, "QSSTRESC", "1"],
["QS", "01-701-1181",4059, "QSSTRESC", "1"],
["QS", "01-701-1181",4020, "QSSTRESC", "1"],
["QS", "01-701-1023",1010, "QSSTRESN", 2],
["QS", "01-701-1023",1012, "QSSTRESN", 2],
["QS", "01-701-1111",5004, "QSSTRESN", 4],
["QS", "01-701-1111",5019, "QSSTRESN", 4],
["QS", "01-701-1111",5012, "QSSTRESN", 4],
["QS", "01-701-1111",5027, "QSSTRESN", 4],
["QS", "01-701-1118",6002, "QSSTRESN", 1],
["QS", "01-701-1118",6003, "QSSTRESN", 7],
["QS", "01-701-1181",4018, "QSSTRESN", 1],
["QS", "01-701-1181",4058, "QSSTRESN", 1],
["QS", "01-701-1181",4019, "QSSTRESN", 1],
["QS", "01-701-1181",4059, "QSSTRESN", 1],
["QS", "01-701-1181",4020, "QSSTRESN", 1],
["QS", "01-701-1118",6001, "QSDTC", "2014-07-08"],
["QS", "01-701-1118",6001, "QSDY", 119],
["AE", "01-701-1015",   3, "AESER", "Y"],
["AE", "01-701-1015",   3, "AESHOSP", "Y"],
["AE", "01-701-1015",   3, "AESTDTC", "2014-01-11"],
["AE", "01-701-1015",   3, "AEENDTC", "2014-01-09"],
["AE", "01-701-1015",   3, "AESTDY", 10],
["AE", "01-701-1015",   3, "AEENDY", 8],
["AE", "01-701-1028",   1, "AETERM", "PARKINSON'S DISEASE"],
["AE", "01-701-1028",   1, "AESTDTC", "2013-07-01"],
["AE", "01-701-1028",   1, "AESTDY", -17],
["AE", "01-701-1034",   2, "AETERM", "MALIGNANT HYPERTENSION"],
["AE", "01-701-1047",   4, "AETERM", "HYPERTENSION"],
["AE", "01-701-1363",   1, "AESTDTC", "2013-06-15"],
["AE", "01-701-1363",   1, "AEENDTC", "2013-06-14"],
["AE", "01-701-1363",   1, "AESTDY", 17],
["AE", "01-701-1363",   1, "AEENDY", 16],
["AE", "01-701-1047",   3, "AEENDTC", "2013-03-05"],
["AE", "01-701-1047",   3, "AEENDY", 22],
["AE", "01-701-1383",  12, "AETERM", "BLOOD PRESSURE INCREASED"],
["AE", "01-701-1153",   2, "AEACN", "DRUG WITHDRAWN"],
["AE", "01-701-1180",   6, "AETERM", "SUDDEN DEATH"],
["AE", "01-703-1258",   2, "AESEV", "SEVERE"],
["AE", "01-703-1258",   2, "AESTDTC", "2012-08-01"],
["AE", "01-703-1258",   2, "AEENDTC", "2012-10-01"],
["AE", "01-703-1258",   2, "AESTDY", 13],
["AE", "01-703-1258",   2, "AEENDY", 74],
["AE", "01-703-1258",   5, "AESEV", "MODERATE"],
["AE", "01-703-1258",   5, "AESER", "Y"],
["AE", "01-703-1258",   5, "AEOUT", "RECOVERED/RESOLVED"],
["AE", "01-703-1258",   5, "AESLIFE", "Y"],
["AE", "01-703-1258",   5, "AESTDTC", "2012-10-02"],
["AE", "01-703-1258",   5, "AEENDTC", "2012-12-31"],
["AE", "01-703-1258",   2, "AESTDY", 75],
["AE", "01-703-1258",   2, "AEENDY", 165],
["AE", "01-703-1335",   1, "AETERM", "MULTIPLE SCLEROSIS RELAPSE"],
["AE", "01-703-1335",   1, "AESTDTC", "2014-04-01"],
["AE", "01-703-1335",   1, "AEENDTC", "2014-05-01"],
["AE", "01-703-1335",   1, "AESTDY", 15],
["AE", "01-703-1335",   1, "AEENDY", 46],
["AE", "01-703-1403",   2, "AETERM", "MYASTHENIA GRAVIS AGGRAVATED"],
["AE", "01-704-1008",   1, "AETERM", "TREMOR IN HANDS, LEGS"],
["AE", "01-704-1008",   1, "AEREL", "NONE"],
["AE", "01-704-1008",   1, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   1, "AESTDY", -225],
["AE", "01-704-1008",   3, "AETERM", "MUSCLE STIFFNESS"],
["AE", "01-704-1008",   3, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   3, "AESTDY", -225],
["AE", "01-704-1008",   2, "AETERM", "SLOWNESS of MOVEMENT"],
["AE", "01-704-1008",   2, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   2, "AESTDY", -225],
["AE", "01-704-1009",   6, "AETERM", "CHRONIC KIDNEY DISEASE"],
["AE", "01-704-1009",   6, "AESER", "Y"],
["AE", "01-704-1009",   6, "AESLIFE", "Y"],
["AE", "01-704-1010",   1, "AETERM", "DIABETES MELLITUS"],
["AE", "01-704-1010",   1, "AESER", "Y"],
["AE", "01-704-1010",   1, "AESLIFE", "Y"],
["AE", "01-704-1017",   4, "AETERM", "LATE EFFECTS OF CEREBRAL INFRACTION"],
["AE", "01-704-1017",   4, "AESEV", "SEVERE",],
["AE", "01-704-1017",   4, "AESTDTC", "2013-10-19"],
["AE", "01-704-1017",   4, "AEENDTC", "2013-11-18"],
["AE", "01-704-1017",   4, "AESTDY", 14],
["AE", "01-704-1017",   4, "AEENDY", 44],
["AE", "01-704-1017",   3, "AETERM", "BRAIN DEATH"],
["AE", "01-704-1017",   3, "AESEV", "SEVERE",],
["AE", "01-704-1017",   3, "AESTDTC", "2013-11-18"],
["AE", "01-704-1017",   3, "AEENDTC", "2013-11-18"],
["AE", "01-704-1017",   3, "AESTDY", 44],
["AE", "01-704-1017",   3, "AEENDY", 44],
["AE", "01-704-1017",   1, "AEOUT", "RECOVERED/RESOLVED"],
["AE", "01-704-1017",   1, "AESTDTC", "2013-10-19"],
["AE", "01-704-1017",   1, "AEENDTC", "2013-11-19"],
["AE", "01-704-1017",   1, "AESTDY", 14],
["AE", "01-704-1017",   1, "AEENDY", 45],
["AE", "01-704-1017",   1, "AEACN", "DRUG WITHDRAWN"]
]

dataset_list_updated = dataset_list

for l in Target_data:
  #print(l)
  dataset_list_updated = data_update(dataset_list_updated, l[0], l[1], l[2], l[3], l[4])

USUBJID '01-703-1096' の 'AGE' を '49' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '3' の 'LBORRES' を '135' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '4' の 'LBORRES' を '145' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '37' の 'LBORRES' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '72' の 'LBORRES' を '1.2' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '102' の 'LBORRES' を '1.1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '132' の 'LBORRES' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '162' の 'LBORRES' を '1.3' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '197' の 'LBORRES' を '0.9' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '232' の 'LBORRES' を '0.8' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '3' の 'LBSTRESC' を '135' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '4' の 'LBSTRESC' を '145' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '37' の 'LBSTRESC' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '72' の 'LBSTRESC' を '1.2' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '102' の 'LBSTRESC' を '1.1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '132' の 'LBSTRESC' を '1' に更

In [33]:
# 更新データ確認
compare_data(dataset_list, dataset_list_updated)

with open("dataset_list_updated.json", "w") as f:
  json.dump(dataset_list_updated, f)

--- ItemGroupOID: AE ---
USUBJID = 01-701-1015, AESEQ = 3:
  Updated:
    AEENDTC: '2014-01-11' -> '2014-01-09'
    AEENDY: 10 -> 8
    AESER: 'N' -> 'Y'
    AESHOSP: 'N' -> 'Y'
    AESTDTC: '2014-01-09' -> '2014-01-11'
    AESTDY: 8 -> 10

USUBJID = 01-701-1028, AESEQ = 1:
  Updated:
    AESTDTC: '2013-07-21' -> '2013-07-01'
    AESTDY: 3 -> -17
    AETERM: 'APPLICATION SITE ERYTHEMA' -> "PARKINSON'S DISEASE"

USUBJID = 01-701-1034, AESEQ = 2:
  Updated:
    AETERM: 'FATIGUE' -> 'MALIGNANT HYPERTENSION'

USUBJID = 01-701-1047, AESEQ = 3:
  Updated:
    AEENDTC: '' -> '2013-03-05'
    AEENDY: None -> 22

USUBJID = 01-701-1047, AESEQ = 4:
  Updated:
    AETERM: 'BUNDLE BRANCH BLOCK LEFT' -> 'HYPERTENSION'

USUBJID = 01-701-1153, AESEQ = 2:
  Updated:
    AEACN: '' -> 'DRUG WITHDRAWN'

USUBJID = 01-701-1180, AESEQ = 6:
  Updated:
    AETERM: 'MICTURITION URGENCY' -> 'SUDDEN DEATH'

USUBJID = 01-701-1363, AESEQ = 1:
  Updated:
    AEENDTC: '2013-06-15' -> '2013-06-14'
    AEENDY: 17 -> 16

# LLMへの送信

In [34]:
!pip install sseclient-py
import requests
import sseclient
from IPython.display import display, Markdown

from google.colab import userdata
api_key = userdata.get('Dify_DatasetJSON')
user_id = 'JPMA_Sample'

## ワークフロー実行関数定義

In [35]:
import json
import time
import requests
import sseclient

# 定数
DIFY_API_URL = 'https://api.dify.ai/v1/workflows/run'
CONTENT_TYPE_JSON = 'application/json'

def call_dify_api(api_key: str, payload: dict, stream: bool = False) -> requests.Response:
    """Dify APIを呼び出す共通関数"""
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': CONTENT_TYPE_JSON
    }
    try:
        response = requests.post(DIFY_API_URL, headers=headers, json=payload, stream=stream)
        response.raise_for_status()  # HTTPエラーが発生した場合に例外を発生させる
        return response
    except requests.exceptions.RequestException as e:
        print(f"API呼び出しエラー: {e}")
        raise

def run_dify_workflow(api_key: str, workflow_inputs: dict, user_id: str, streaming: bool = False) -> dict | sseclient.Event:
    """Difyワークフローを実行する

    Args:
        api_key: Dify APIキー
        workflow_inputs: ワークフローへの入力
        user_id: ユーザーID
        streaming: ストリーミングモードで実行するかどうか (Falseの場合はブロッキングモード)

    Returns:
        ストリーミングモードの場合はsseclient.Eventのイテレータ、
        ブロッキングモードの場合はAPIのレスポンスのJSONを辞書型で返す
    """
    payload = {
        'inputs': workflow_inputs,
        'response_mode': 'streaming' if streaming else 'blocking',
        'user': user_id
    }
    response = call_dify_api(api_key, payload, stream=streaming)
    if streaming:
        client = sseclient.SSEClient(response)
        return client.events()
    else:
        return response.json()

def safe_print_event_data(event: sseclient.Event):
    """
    与えられたSSEイベントデータから、存在する場合に特定の値を出力します。
    キーが存在しない場合は何も出力しません。Statusが存在する場合にのみTitleと結合させて表示します。

    Args:
        event: イベントデータを含むSSEイベントオブジェクト。event.data属性がJSON文字列であることを想定。
    """
    try:
        data = json.loads(event.data)

        if 'event' in data:
            print(f"Event: {data['event']}")

        if 'data' in data:
            title = data['data'].get('title')
            status = data['data'].get('status')
            if title is not None and status is not None:
                print(f"Node: {title} ({status})")
            elif title is not None:
                print(f"Node: {title}") # Statusが存在しない場合はTitleのみ表示

            if 'error' in data['data']:
                print(f"Error: {data['data']['error']}")
            if 'elapsed_time' in data['data']:
                print(f"Elapsed time: {data['data'].get('elapsed_time')}")
            if 'total_tokens' in data['data']:
                print(f"Total tokens: {data['data'].get('total_tokens')}")

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON event data: {e}")
    except AttributeError as e:
        print(f"Error accessing event data attribute: {e}")

def run_workflow_with_retry(api_key: str, workflow_inputs: dict, user_id: str, max_retries: int = 3, retry_delay: int = 20):
    """ワークフローを実行し、エラー発生時にリトライを行う (ストリーミングモード専用)"""
    for retry in range(max_retries + 1):
        print(f"--- 試行回数: {retry + 1} ---")
        success = True
        try:
            for event in run_dify_workflow(api_key, workflow_inputs, user_id, streaming=True):
                safe_print_event_data(event)
                try:
                    event_data = json.loads(event.data)
                    if event_data.get('data', {}).get('error') is not None:
                        print(f"エラーが検出されました: {event_data['data']['error']}")
                        success = False
                        break
                except json.JSONDecodeError:
                    print("JSONデコードエラーが発生しました。")
                    success = False
                    break
                print('------')

            if success:
                print("ワークフローが正常に完了しました。")
                return json.loads(event.data)
            elif retry < max_retries:
                print(f"エラーが発生したため、{retry_delay}秒後に再試行します...")
                time.sleep(retry_delay)
            else:
                print("最大再試行回数に達しました。ワークフローは失敗しました。")
                return False

        except requests.exceptions.RequestException as e:
            print(f"APIリクエスト中にエラーが発生しました: {e}")
            success = False
            if retry < max_retries:
                print(f"{retry_delay}秒後に再試行します...")
                time.sleep(retry_delay)
            else:
                print("最大再試行回数に達しました。ワークフローは失敗しました。")
                return False

## ファイルアップロード関数定義

In [36]:
import requests
import mimetypes

def upload_file_to_dify(api_key: str, file_path: str, user_id: str):
    """
    Difyにファイルをアップロードします。

    Args:
        api_key (str): Dify APIキー。
        file_path (str): アップロードするローカルファイルのパス。
        user_id (str): このファイルを関連付ける一意のエンドユーザー識別子。

    Returns:
        dict: APIからのレスポンス (JSON形式)。成功時はファイル情報が含まれます。
        None: エラーが発生した場合。
    """

    # APIエンドポイント
    upload_url = "https://api.dify.ai/v1/files/upload"

    mime_type, _ = mimetypes.guess_type(file_path)
    print(mime_type)

    try:
        # ファイルをバイナリモードで開く
        with open(file_path, 'rb') as f:
            # 'file'というキーでファイルオブジェクトを渡す
            files = {
                'file': (os.path.basename(file_path), f, mime_type) # (ファイル名, ファイルオブジェクト)
            }

            # POSTリクエストを送信
            response = requests.post(
                upload_url,
                headers = {"Authorization": f"Bearer {api_key}"},
                data={'user': user_id},
                files=files    # アップロードするファイル
            )
            print(files)

            # エラーレスポンスをチェック (4xx, 5xx)
            response.raise_for_status()

            # 成功した場合、JSONレスポンスを返す
            print(f"ファイル '{os.path.basename(file_path)}' のアップロードに成功しました。")
            return response.json()

    except FileNotFoundError:
        print(f"エラー: 指定されたファイルが見つかりません - {file_path}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"APIリクエスト中にエラーが発生しました: {e}")
        # エラーレスポンスの内容を表示しようと試みる
        try:
            print(f"サーバーからのエラー詳細: {response.text}")
        except NameError: # responseオブジェクトが存在しない場合
             pass
        except Exception as detail_e:
             print(f"サーバーからのエラー詳細の取得中に別のエラー: {detail_e}")
        return None
    except Exception as e:
        print(f"予期せぬエラーが発生しました: {e}")
        return None


### ファイルアップロードとワークフローのテスト

In [37]:
#  # サンプルファイルの取得
#  !wget https://github.com/Takumi173/Test/releases/download/testdata/SampleText.txt
#
#  # アップロードしたいファイルのパス
#  file_to_upload = "SampleText.txt"
#
#  # アップロードの実行
#  upload_result = upload_file_to_dify(api_key, file_to_upload, user_id)
#
#  if upload_result:
#      print("\nアップロード結果:")
#      print(upload_result)
#
#      file_id = upload_result.get('id')
#      print(f"ファイルID: {file_id}")    # Workflowに投げるときはこのファイルIDを指定する
#  else:
#      print("\nアップロードに失敗しました。")
#
#
#  # アップロードしたファイルをワークフローに投げる
#  ModelName = 'gemini-2.0-flash'
#  workflow_inputs = {
#          'ModelName': ModelName,
#          'SysPrompt': '',
#          'UserInput': '以下にの内容を要約してください',
#          'AttachedFile': {"type": "document", "transfer_method": "local_file", "upload_file_id": file_id}
#    }
#
#  result = run_workflow_with_retry(api_key, workflow_inputs, user_id)
#  output_Task = result['data']['outputs']['text']
#  display(Markdown(output_Task))

## 出力フォーマットの定義

In [38]:
# Markdown
Markdown_General = '''
**出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。
'''
Taks1_Markdown = Markdown_General+'''
    1. 症例サマリー：[USUBJID]
        *   YYYY年MM月DD日 (Day XX): [有害事象、検査値、バイタルサインなどのイベントを、異常所見を中心に簡潔な文章で記載]

    2. 疑義事項: [あり/なし]
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]
'''

Taks2_Markdown = Markdown_General+'''
    1. 確認した症例：[USUBJID]

    2. 医療機関に問い合わせるクエリ: [あり/なし]
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]

    3. 医療機関に問い合わせない疑義事項: [あり/なし]
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **疑義事項:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]
'''

Taks3_Markdown = Markdown_General+'''
    1. 確認した症例：[USUBJID]

    2. プロトコル逸脱: [あり/なし]
        *   **逸脱No.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **逸脱内容:** [具体的な逸脱内容を簡潔に記述。例：被験者XXXは、プロトコルで規定された投与量を超える量の治験薬を投与された]
            *   **プロトコル該当箇所:** [プロトコルの該当するセクション、ページ番号などを記載]
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]

    3. 医療機関に問い合わせるクエリ: [あり/なし]
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]

'''

# JSON
JSON_General = '''
**出力形式:**

*   すべての出力は指定されるJSON Schemaを用いて出力してください。
*   コードブロックや改行コードは使用せず、"{"で開始し、"}"で終わるJSONオブジェクト形式で出力してください。

**出力言語:**

*   JSONのValueは日本語で出力します

**JSON Schema**
'''

Task1_JSON = JSON_General+'''
{ "name": "Clinical_Review", "description": "Schema for clinical case summaries and associated queries", "strict": true, "schema": { "type": "object", "properties": { "usubjid": { "type": "string", "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)" }, "timeline": { "type": ["array", "null"], "description": "有害事象、検査値、バイタルサインなどの推移を時系列でまとめた症例サマリー", "items": { "type": "object", "properties": { "date": { "type": "string", "description": "Date of the event.  Allows full (YYYY-MM-DD), partial (YYYY-MM), or year-only (YYYY) formats." }, "day": { "type": "integer", "description": "Day relative to study start (Day 1).  Allows negative values for pre-treatment days." }, "details": { "type": "string", "description": "異常所見を中心に簡潔な文章で記載する。正常範囲内の変動は省略可能。" } }, "required": [ "date", "day", "details" ], "additionalProperties": false } }, "queries": { "type": ["array", "null"], "description": "List of queries related to the subject", "items": { "type": "object", "properties": { "query_no": { "type": "integer", "description": "Unique query number" }, "criticality": { "type": "string", "description": "Criticality of the query on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "inquiry": { "type": "string", "description": "医療機関への問い合わせ文面" }, "reason": { "type": "string", "description": "判断理由" }, "variables": { "type": "array", "description": "List of variables and their values related to the query", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } } }, "required": [ "query_no", "criticality", "inquiry", "reason", "variables" ], "additionalProperties": false } } }, "required": [ "usubjid", "timeline", "queries" ], "additionalProperties": false } }
'''

Task2_JSON = JSON_General+'''
{ "name": "_Review", "description": "Schema for clinical case summaries and associated queries", "strict": true, "schema": { "type": "object", "properties": { "usubjid": { "type": "string", "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)" }, "data_issues": { "type": ["array", "null"], "description": "List of data issues identified during review", "items": { "type": "object", "properties": { "issue_no": { "type": "integer", "description": "Unique issue number" }, "variables": { "type": "array", "description": "List of variables and their values related to the issue", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } }, "inconsistency": { "type": "string", "description": "具体的な矛盾の内容を記述" }, "cause": { "type": "string", "description": "問題点の原因（推測）" }, "resolution": { "type": "string", "description": "対応策（提案）" } }, "required": [ "issue_no", "variables", "inconsistency", "cause", "resolution" ], "additionalProperties": false } }, "queries": { "type": ["array", "null"], "description": "List of queries related to the subject", "items": { "type": "object", "properties": { "query_no": { "type": "integer", "description": "Unique query number" }, "criticality": { "type": "string", "description": "Criticality of the query on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "inquiry": { "type": "string", "description": "医療機関への問い合わせ文面" }, "reason": { "type": "string", "description": "判断理由" }, "variables": { "type": "array", "description": "List of variables and their values related to the query", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } } }, "required": [ "query_no", "criticality", "inquiry", "reason", "variables" ], "additionalProperties": false } } }, "required": [ "usubjid", "data_issues", "queries" ], "additionalProperties": false } }
'''

Task3_JSON = JSON_General+'''
{ "name": "Protocol_Deviation_Review", "description": "Schema for clinical case summaries and associated queries", "strict": true, "schema": { "type": "object", "properties": { "usubjid": { "type": "string", "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)" }, "deviations": { "type": ["array", "null"], "description": "List of protocol deviations", "items": { "type": "object", "properties": { "deviation_no": { "type": "integer", "description": "Unique deviation number" }, "impact": { "type": "string", "description": "Impact of the deviation on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "variables": { "type": "array", "description": "List of variables and their values related to the deviation", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } }, "description": { "type": "string", "description": "具体的な逸脱内容を簡潔に記述。" }, "protocol_reference": { "type": "string", "description": "プロトコルの該当するセクション、ページ番号などを記載" }, "justification": { "type": "string", "description": "判断理由" } }, "required": [ "deviation_no", "impact", "variables", "description", "protocol_reference", "justification" ], "additionalProperties": false } }, "queries": { "type": ["array", "null"], "description": "List of queries related to the subject", "items": { "type": "object", "properties": { "query_no": { "type": "integer", "description": "Unique query number" }, "criticality": { "type": "string", "description": "Criticality of the query on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "inquiry": { "type": "string", "description": "医療機関への問い合わせ文面" }, "reason": { "type": "string", "description": "判断理由" }, "variables": { "type": "array", "description": "List of variables and their values related to the query", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } } }, "required": [ "query_no", "criticality", "inquiry", "reason", "variables" ], "additionalProperties": false } } }, "required": [ "usubjid", "deviations", "queries" ], "additionalProperties": false } }
'''


## プロンプトの作成

In [39]:
with open('define_xml/define.xml', 'r') as f:
  define_xml = f.read()


SysPrompt = '''
あなたは、臨床試験データのレビューを支援するAIアシスタントです。以下の前提知識を理解した上で、ユーザーからの指示（ユーザープロンプト）に従って、臨床試験データのレビューを支援してください。各タスクでは、ユーザープロンプトで指定された役割になりきって回答してください。

**前提知識:**

*   臨床試験においては患者の安全性が最優先され、有害事象の評価は特に重要です。
*   SDTM (Study Data Tabulation Model) は、CDISCによって策定された臨床試験データの標準モデルです。
*   Define.xmlはSDTMデータの構造を記述したメタデータファイルであり、参考情報として使用します。JSONデータ自体の内容、医学的妥当性、プロトコルとの整合性を優先してレビューしてください。
*   SDTMデータは、DM、AE、VS、LBなど、複数のドメイン（データセット）に分かれています。
*   報告されるJSONデータには、データ入力時の間違いが含まれる可能性があります。
*   提供された情報のみに基づいて回答を作成してください。想像やハルシネーションに基づいた回答は作成してはいけません。

**その他:**

*   指定された出力フォーマットに厳密に従って出力してください。
*   JSONデータまたはDefine.xmlの形式が不正な場合は、その旨をエラーメッセージとして出力してください。
'''




UserInput_Task1 = '''
あなたは臨床試験の専門医です。以下の指示に従い、提供される情報（プロトコル、JSONデータ、Define.xml）を基に、臨床試験データのレビューとクエリ作成（必要な場合）を行ってください。

**1. 症例サマリーの作成:**

*   **参照情報:** JSONデータ、Define.xml
*   **タスク:**
    *   JSONデータとDefine.xmlを参照し、有害事象、検査値、バイタルサインなどの推移を時系列でまとめた症例サマリーを作成してください。
    *   特に、**異常所見**を中心に簡潔な文章で記載してください。正常範囲内の変動は省略して構いません。
    *   各イベントの日時は、Define.xmlに定義された日付変数などを参考に、正確に特定してください。

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   以下のJSONデータのレビュー観点に基づき、JSONデータを改めて点検してください。
    *   医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、臨床試験の評価項目に対する影響度を考慮し、重要度の高いものから優先的に作成してください。
    *  **疑義事項がない場合は、クエリを作成する必要はありません。**「疑義事項なし」と回答してください。

*   **JSONデータのレビュー観点 (これらに限定されない):**
    *   **安全性:** 有害事象(AEドメイン)の報告内容は、医学的に妥当であるか？
    *   **医学的妥当性:** 検査値(LBドメイン)の変動、バイタルサイン(VSドメイン)の変動、併用薬(CMドメイン)との相互作用など、時間経過とともに医学的に問題となる点は見られるか？
    *   **有効性:** 特定された主要評価項目および副次評価項目について、その時間的変化は期待される効果と一致しているか？
    *   **その他:** 患者背景(DMドメイン)、既往歴(MHドメイン)、有害事象(AEドメイン)、治療歴(EXドメイン, CMドメイン)などを総合的に考慮し、時間経過を加味して安全性に懸念を生じる事項があれば記載してください。
    *   **プロトコル逸脱 (疑い):** 選択/除外基準、投与量、併用禁止薬、評価スケジュール、有害事象報告などについて、プロトコルからの逸脱の疑いがないか確認してください。（関連ドメイン: DM, MH, EX, CM, LB, VS, AEなど）
'''



UserInput_Task2 = '''
あなたはクリニカルデータマネージャーです。以下の指示に従い、提供される情報（JSONデータ、Define.xml、プロトコル）を基に、データ整合性レビューとクエリ作成（必要な場合）を行ってください。

**1. データ整合性レビュー:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   JSONデータ、Define.xml、プロトコルを参照し、データの不整合が疑われる問題点を検出してください。
    *   **特に、以下の点に焦点を当ててレビューしてください。**
        *   **クロスドメイン整合性:** 異なるSDTMドメイン間で、データに矛盾がないか、ドメイン間の関連性が正しく表現されているか。
            *   **具体的な確認例 (これらに限定されない):**
                *   DM.SEXとAEにおける妊娠関連の有害事象
                *   AEの有害事象発現日や治験薬との関連性と、EXの治験薬の投与期間
                *   LBの検査値異常とAEの関連有害事象
                *   VSのバイタルサイン異常とAEの関連有害事象
                *   CM.CMTRTとAE/MHで報告されている疾患・既往歴との矛盾

        *   **単一ドメイン内の整合性:** Define.xmlの定義に照らして、矛盾なく解釈できるデータになっているか、プロトコルに照らしてデータの関連性が正しく表現されているか。
        *   **異常値:** Define.xmlで定義された範囲外、または医学的にありえない値がないか。
        *   **欠損値:** 欠損値の有無と理由（推測できる場合）。多い場合は原因を推測。
        *   **プロトコル逸脱 (データ品質の観点から):** データ入力/収集で、プロトコルからの逸脱（例：必須項目の未入力、不適切な時期のデータ収集）がないか。

    *   Define.xmlとデータの間に不整合がある場合は、「Define.xmlの修正候補」として報告してください。

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   データ整合性レビューの結果、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、臨床試験の評価項目に対する影響度を考慮し、重要度の高いものから優先的に作成してください。
    *   **疑義事項がない場合は、クエリを作成する必要はありません。**
'''



UserInput_Task3 = '''
あなたは、臨床試験の専門医、データマネージャー、CRAの視点を持つ、プロトコル遵守状況の確認者です。以下の指示に従い、提供される情報（JSONデータ、Define.xml、プロトコル）を基に、プロトコル逸脱の検出とクエリ作成（必要な場合）を行ってください。

**1. プロトコル逸脱の検出:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   JSONデータ、Define.xml、プロトコルを参照し、プロトコルからの逸脱を検出してください。
    *   Define.xmlは参考情報として活用し、データとプロトコルの内容を比較して逸脱を判断してください。
    *   **検出対象とすべき主要なプロトコル逸脱の例 (これらに限定されない):**
        *   **選択/除外基準違反:** (関連SDTMドメイン: DM, MH など)
        *   **投与量違反:** (関連SDTMドメイン: EX)
        *   **併用禁止薬の使用:** (関連SDTMドメイン: CM)
        *   **評価スケジュール違反:** (関連SDTMドメイン: LB, VS, その他)
        *   **有害事象報告違反**: (関連SDTMドメイン: AE)

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   プロトコル逸脱を判定するために、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、プロトコル逸脱が臨床試験の評価項目に与える影響度を考慮し、重要度の高いものから優先的に作成してください。
    *   **プロトコル逸脱に関する疑義事項がない場合は、クエリを作成する必要はありません。**
'''


UserInput_end1 = '''\n---\n\n**データ:**\n\n*   臨床試験データ（JSON形式、SDTM準拠）:\n\n```json\n'''
UserInput_end2 = '''\n```\n\n*   データ定義ファイル（Define.xml）:\n\n```xml\n''' + define_xml + '''```\n'''

In [56]:
SysPrompt_single = '''
あなたは、臨床試験データの正確性、完全性、医学的妥当性、プロトコル遵守状況のレビューを支援するAIアシスタントです。データマネージャー、メディカルモニター、臨床専門医などの視点を持ち、ユーザーからの指示（ユーザープロンプト）に従って、臨床試験データのレビューを支援します。

**最重要原則:**
*   **提供された情報（JSONデータ、Define.xml、プロトコル）のみ**に基づいて、客観的な事実に基づいた回答を作成してください。
*   **いかなる場合も、想像、推測、ハルシネーションに基づいた情報を生成してはいけません。** 根拠のない情報や、提供された情報からは導き出せない結論を提示しないでください。
*   患者の安全性を最優先し、有害事象の評価には特に注意を払ってください。

**前提知識:**
*   **SDTM (Study Data Tabulation Model):** CDISCによって策定された臨床試験データの標準モデルです。データはDM, AE, VS, LBなどのドメインに分かれており、レビューにはこれらの**ドメイン情報を横断的・統合的に評価する**必要があります。
*   **Define.xml:** SDTMデータの構造（変数名、ラベル、コードリスト、データ型など）を記述したメタデータファイルであり、データの意味を正確に理解するために**不可欠な情報源**です。JSONデータの解釈は、**必ずDefine.xmlの定義に基づいて**行ってください。
*   **データの不完全性:** 報告されるJSONデータには、データ入力時の間違いや不整合が含まれる可能性があることを理解しています。
*   **プロトコル:** 臨床試験の実施計画書であり、選択/除外基準、投与計画、評価スケジュール、有害事象報告手順などが規定されています。データのレビューはプロトコル遵守の観点からも行います。

**タスク実行における注意:**
*   指定された**出力フォーマット**に厳密に従ってください。
*   提供された情報からタスクを実行できない場合（例：必要な情報が欠けている、矛盾が解決できない）、その旨を明確に指摘してください。

**エラーハンドリング:**
*   JSONデータ、Define.xml、またはプロトコルの形式が不正である、あるいは内容が著しく不足しておりレビューが困難な場合は、具体的な問題点を指摘し、処理を中断してください。例：「エラー：Define.xmlファイルが提供されていません。」、「エラー：JSONデータの[ドメイン名]に必要な変数[変数名]が含まれていません。」
'''

UserInput_single = '''
**役割:**

あなたは、**臨床試験データの多角的なレビュー担当者**です。**臨床専門医、クリニカルデータマネージャー、およびプロトコル遵守確認者の視点を併せ持ち**、以下の指示に従って、提供される情報（プロトコル、JSONデータ、Define.xml）を基に、臨床試験データの統合レビュー、疑義事項の特定、およびクエリ/内部確認事項の作成（必要な場合）を行ってください。

**指示:**

**1. 症例サマリーの作成:**

*   **参照情報:** JSONデータ、Define.xml
*   **タスク:**
    *   JSONデータとDefine.xmlを参照し、患者の主要なイベントを時系列でまとめたサマリーを作成してください。
    *   **患者背景:** 最初にDMドメインから、主要な背景情報（例: 年齢、性別、人種など、Define.xmlで定義されたラベルを使用）を記載してください。
    *   **イベント推移:** 有害事象(AE)、検査値(LB)、バイタルサイン(VS)について、**異常変動**や**臨床的に注目すべき変化**を中心に記述してください。
        *   **異常・注目すべき変化の基準(例):**
            *   有害事象の発現、重症度・重篤度の変化、転帰
            *   検査値・バイタルサインの基準値からの逸脱 (Grade変化や明らかな異常値)
            *   ベースラインからの著しい変動
            *   正常範囲上限/下限付近での臨床的に意味のある変動
        *   **省略:** 原則として正常範囲内で臨床的に意義の小さい変動は省略してください。
    *   **日時:** 各イベントの日時は、関連する日付変数（例：AESTDY, LBDY, VSDYなど、Define.xml参照）に基づき特定し、**Study Day (--DY) を括弧内に併記**してください (例: `(Day 10)`)。
    *   **記述:** 簡潔な文章で客観的に記述してください。

**2. 統合レビュー:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   以下の**統合レビュー観点**に基づき、提供された情報を注意深く照合し、JSONデータを多角的にレビューしてください。問題点、矛盾、不整合、プロトコルからの逸脱の可能性などを、臨床試験の評価項目や患者の安全性への影響度が高い順に検出・指摘してください。

*   **統合レビュー観点:**
    *   **【医学的妥当性・安全性】 (専門医視点)**
        *   **AE:** 有害事象の報告内容（事象名、重篤度、重症度、治験薬との関連性、処置、転帰）に、他のデータ（LB, VS, CMなど）との矛盾や医学的な観点から不自然な点はないか？特に重篤な有害事象の評価は、関連データと照らして一貫性があるか？
        *   **LB/VS:** 検査値やバイタルサインの変動パターン（異常値、経時変化）に、医学的に懸念される点はないか？ 他の臨床情報（AE, CM, MHなど）と整合しているか？
        *   **CM/AE/MH:** 併用薬と有害事象/既往歴との関連、潜在的な薬物相互作用に関して、プロトコルや一般的な医学知識に基づき、注意すべき点はないか？
        *   **総合評価:** 患者背景(DM)、既往歴(MH)、有害事象(AE)、治療歴(EX, CM)などを総合的に考慮し、時間経過を踏まえて、患者の安全性に関する潜在的な懸念事項はないか？

    *   **【データ整合性】 (データマネージャー視点)**
        *   **クロスドメイン整合性:** 異なるドメイン間のデータに矛盾はないか？ (例: AE発生日 vs LB/VS測定日、AE回復日 vs LB/VS測定日、AE vs CM開始/終了日、MH vs AE/CM、DM.SEX vs 性別依存のイベント/検査、AE発現日 vs EX投与期間)
        *   **ドメイン内整合性:** 各ドメイン内のデータに矛盾はないか？ (例: AE 開始日 <= AE 終了日、投与量と単位の一貫性)
        *   **異常値/外れ値:** 医学的/現実的にありえない値、Define.xmlで定義された範囲外の値はないか？
        *   **欠損値:** 重要な変数に欠損はないか？ (例: AE関連性、LB/VS結果、主要評価項目) 欠損が許容されるか、理由が適切か？

    *   **【プロトコル遵守】 (プロトコル確認者視点)**
        *   **選択/除外基準:** 患者はプロトコルで規定された選択基準を満たし、除外基準に該当していないか？ (DM, MHなどを参照)
        *   **治験薬投与:** 投与量、投与経路、投与期間などはプロトコルで規定された通りか？ (EXを参照)
        *   **併用禁止/制限薬:** プロトコルで禁止または制限されている薬剤が使用されていないか？ (CMを参照)
        *   **評価スケジュール/手順:** 検査や評価はプロトコルで規定されたタイミングと手順で実施されているか？ (VISIT情報、各ドメインの--DY/VISITNUMなどを参照)
        *   **有害事象報告:** 有害事象（特に重篤な有害事象）はプロトコルの規定に従って報告されているか？ (AEを参照)

**3. 疑義事項の分類とクエリ/内部確認事項の作成:**

*   **参照情報:** 統合レビューの結果、JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   **統合レビューで特定された問題点や疑義事項についてのみ**、以下の分類を行ってください。
        *   **医療機関へのクエリ:** 医療機関への問い合わせが必要な事項。
        *   **内部確認事項:** 医療機関への問い合わせは不要だが、内部で確認・記録すべき事項（例：軽微なデータ不整合、解釈の確認）。

    *   作成する際は、**必ず提供された情報（JSONデータ、Define.xml、プロトコル）に基づく客観的な事実のみを根拠**としてください。**推測やハルシネーションに基づく記述は絶対に含めないでください。**
    *   指摘事項とクエリ/内部確認事項は、臨床試験の評価項目や患者の安全性への影響度を考慮し、重要度（Critical/Major/Minorのいずれか）を付与してください。
    *   **レビューの結果、クエリや内部確認事項を作成する必要がない場合は、「疑義事項なし」と明確に回答してください。**

**出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。

```markdown
# 臨床試験データ統合レビュー報告

## 1. 症例サマリー：[USUBJID]

*   **患者背景:** [DMドメインから取得した主要な背景情報を簡潔に記載]
*   **イベント推移:**
    *   [YYYY年MM月DD日 (Day XX): イベント内容 (例: 有害事象「頭痛」(Grade 2) 発現)]
    *   [YYYY年MM月DD日 (Day YY): イベント内容 (例: ALT値上昇 (Grade 1, 基準値上限の1.5倍))]
    *   ... (時系列で記載) ...

## 2. 統合レビュー結果

*   **医学的観点からの指摘事項:**
    *   [指摘事項がない場合は「指摘事項なし」と記載]
    *   (指摘事項がある場合)
        *   **指摘No.:** M-1
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **内容:** [具体的な医学的懸念事項や妥当性に関する指摘]
            *   **根拠:** [判断の根拠となった変数名 = 値、プロトコルの記述などを記載]
        *   ... (複数の指摘事項があれば M-2, M-3...)

*   **データ整合性観点からの指摘事項:**
    *   [指摘事項がない場合は「指摘事項なし」と記載]
    *   (指摘事項がある場合)
        *   **指摘No.:** D-1
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **内容:** [具体的なデータの不整合、異常値、欠損値などに関する指摘]
            *   **根拠:** [判断の根拠となった変数名 = 値、Define.xmlの記述などを記載]
            *   **(Define.xml修正候補):** [必要であれば記載]
        *   ... (複数の指摘事項があれば D-2, D-3...)

*   **プロトコル遵守観点からの指摘事項 (逸脱の可能性):**
    *   [指摘事項がない場合は「指摘事項なし」と記載]
    *   (指摘事項がある場合)
        *   **指摘No.:** P-1
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **逸脱の可能性:** [具体的なプロトコルからの逸脱の可能性]
            *   **プロトコル該当箇所:** [プロトコルの該当するセクション、ページ番号などを記載]
            *   **根拠:** [判断の根拠となった変数名 = 値などを記載]
        *   ... (複数の指摘事項があれば P-2, P-3...)

## 3. 疑義事項

*   [クエリも内部確認事項もない場合は「疑義事項なし」と記載]
*   **医療機関へのクエリ:**
    *   [クエリがない場合は「クエリなし」と記載]
    *   (クエリがある場合)
        *   **クエリNo.:** Q-1 (関連指摘No.: [例: M-1, D-2])
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:** [具体的かつ客観的な問い合わせ内容。例: "Day 10の有害事象「XXXX」の重症度(AE.AESEV)がGrade 2と報告されていますが、同日の臨床検査(LB)では基準値の範囲内です。XXXXの重症度および転機についてご確認ください。"]
            *   **判断理由:** [なぜ問い合わせが必要かの簡潔な理由。例: 有害事象の評価と関連する処置の整合性を確認するため。]
            *   **判断根拠:**
                *   [変数名 = 値; 例: AE.AETERM = '頭痛', AE.AESTDY = 10, AE.AESEV = 'MODERATE (Grade 2)']
                *   [変数名 = 値; 例: CM.CMTRT where CMSTDY = 10 (該当レコードなし)]
                *   [プロトコル該当箇所: 必要であれば記載]
        *   ... (複数のクエリがあれば Q-2, Q-3...)

*   **内部確認事項 (問い合わせ不要):**
    *   [内部確認事項がない場合は「内部確認事項なし」と記載]
    *   (内部確認事項がある場合)
        *   **確認事項No.:** I-1 (関連指摘No.: [例: D-1])
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **疑義事項/確認内容:** [問い合わせは不要だが、記録・確認すべき内容。例: DMドメインの生年月日(BRTHDTC)が一部不明瞭('19XX')だが、年齢(AGE)は計算されており、選択基準を満たしているため現時点では問い合わせ不要と判断。記録として残す。]
            *   **判断理由:** [なぜ問い合わせ不要か、なぜ記録が必要かの簡潔な理由。例: 年齢情報は他の変数で補完されており、選択基準の確認は可能。データの完全性の観点から記録。]
            *   **判断根拠:**
                *   [変数名 = 値; 例: DM.USUBJID = 'XXX', DM.BRTHDTC = '19XX', DM.AGE = 55]
                *   [プロトコル該当箇所: 例: Section 4.1 選択基準 (年齢 18-75歳)]
                *   [Define.xml該当箇所: 必要であれば記載]
            *   ... (複数の内部確認事項があれば I-2, I-3...)

```
'''

UserInput_single_end1 = '''\n---\n\n**臨床試験データ（JSON形式、SDTM準拠）:**\n\n```json\n'''
UserInput_single_end2 = '''\n```\n\n**データ定義ファイル（Define.xml）:**\n\n```xml\n''' + define_xml + '''```\n'''

In [41]:
def create_workflow_input(ModelName, SysPrompt, UserInput_Task, UserInput_end1, datasetjson, UserInput_end2):
    return {
        'ModelName': ModelName,
        'SysPrompt': SysPrompt,
        'UserInput': UserInput_Task + UserInput_end1 + datasetjson + UserInput_end2,
        'AttachedFile': {"type": "document", "transfer_method": "local_file", "upload_file_id": "6b06d4f8-d47a-441f-bf67-d8700f76f556"}
    }

In [42]:
def create_workflow_input_single(ModelName, SysPrompt_single, UserInput_single, UserInput_single_end1, datasetjson, UserInput_single_end2):
    return {
        'ModelName': ModelName,
        'SysPrompt': SysPrompt,
        'UserInput': UserInput_single + UserInput_single_end1 + datasetjson + UserInput_single_end2,
        'AttachedFile': {"type": "document", "transfer_method": "local_file", "upload_file_id": "6b06d4f8-d47a-441f-bf67-d8700f76f556"}
    }

## 実行

In [47]:
# ModelNameの設定
#ModelName = 'gemini-2.0-flash'
#ModelName = 'gemini-2.0-flash-exp'
#ModelName = 'gemini-2.0-flash-exp-multi'
ModelName = 'gemini-2.0-pro-exp'
#ModelName = 'gemini-2.0-pro-exp-02-05'
#ModelName = 'gemini-2.0-flash-thinking-exp-01-21'
#ModelName = 'gemini-2.0-flash-thinking-exp-01-21-multi'
#ModelName = 'gemini-2.0-flash-thinking-exp'
#ModelName = 'gemini-2.0-flash-thinking-exp-multi'


# データ更新症例の抽出
updated_subjects = []
for l in Target_data:
  updated_subjects.append(l[1])

updated_subjects = sorted(list(set(updated_subjects)))
print(updated_subjects)
print(len(updated_subjects))


['01-701-1015', '01-701-1023', '01-701-1028', '01-701-1034', '01-701-1047', '01-701-1097', '01-701-1111', '01-701-1118', '01-701-1146', '01-701-1148', '01-701-1153', '01-701-1180', '01-701-1181', '01-701-1363', '01-701-1383', '01-701-1387', '01-702-1082', '01-703-1042', '01-703-1076', '01-703-1086', '01-703-1096', '01-703-1258', '01-703-1279', '01-703-1299', '01-703-1335', '01-703-1403', '01-704-1008', '01-704-1009', '01-704-1010', '01-704-1017']
30


In [48]:
# output mode: Markdown / JSON
Output_Format = "Markdown"

if Output_Format == "Markdown":
  UserInput_Task1 = UserInput_Task1 + Taks1_Markdown
  UserInput_Task2 = UserInput_Task2 + Taks2_Markdown
  UserInput_Task3 = UserInput_Task3 + Taks3_Markdown
elif Output_Format == "JSON":
  UserInput_Task1 = UserInput_Task1 + Task1_JSON
  UserInput_Task2 = UserInput_Task2 + Task2_JSON
  UserInput_Task3 = UserInput_Task3 + Task3_JSON


In [57]:
import pandas as pd

results_list = []

for subj in updated_subjects[29:30]:
    datasetjson = filter_data(dataset_list_updated, subj)
    print(f"処理完了：'datasetjson' に USUBJID が {subj} のデータを出力しました。")

    row_data = {'Subject': subj}  # 各行のデータを格納する辞書

#    # Task 1 の処理
#    workflow_inputs_Task1 = create_workflow_input(ModelName, SysPrompt, UserInput_Task1, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
#    try:
#        result_Task1 = run_workflow_with_retry(api_key, workflow_inputs_Task1, user_id)
#        output_Task1 = result_Task1['data']['outputs']['text']
#        display(Markdown(output_Task1))
#        row_data['Task1'] = output_Task1
#    except Exception as e:
#        print(f"Task 1 でエラーが発生しました (Subject: {subj}): {e}")
#        row_data['Task1'] = "Error"
#
#    # Task 2 の処理
#    workflow_inputs_Task2 = create_workflow_input(ModelName, SysPrompt, UserInput_Task2, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
#    try:
#        result_Task2 = run_workflow_with_retry(api_key, workflow_inputs_Task2, user_id)
#        output_Task2 = result_Task2['data']['outputs']['text']
#        display(Markdown(output_Task2))
#        row_data['Task2'] = output_Task2
#    except Exception as e:
#        print(f"Task 2 でエラーが発生しました (Subject: {subj}): {e}")
#        row_data['Task2'] = "Error"
#
#    # Task 3 の処理
#    workflow_inputs_Task3 = create_workflow_input(ModelName, SysPrompt, UserInput_Task3, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
#    try:
#        result_Task3 = run_workflow_with_retry(api_key, workflow_inputs_Task3, user_id)
#        output_Task3 = result_Task3['data']['outputs']['text']
#        display(Markdown(output_Task3))
#        row_data['Task3'] = output_Task3
#    except Exception as e:
#        print(f"Task 3 でエラーが発生しました (Subject: {subj}): {e}")
#        row_data['Task3'] = "Error"
#
    # 統合プロンプトの処理
    workflow_inputs_single = create_workflow_input_single(ModelName, SysPrompt_single, UserInput_single, UserInput_single_end1, json.dumps(datasetjson), UserInput_single_end2)
    try:
        result_Task_single = run_workflow_with_retry(api_key, workflow_inputs_single, user_id)
        output_Task_single = result_Task_single['data']['outputs']['text']
        display(Markdown(output_Task_single))
        row_data['Task_single'] = output_Task_single
    except Exception as e:
        print(f"統合プロンプトの処理でエラーが発生しました (Subject: {subj}): {e}")
        row_data['Task_single'] = "Error"
    results_list.append(row_data)

# DataFrameを作成
df_results = pd.DataFrame(results_list)

# DataFrameを表示
display(df_results)

警告：データセット 'CDISCPILOT01.te' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ts' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ti' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.tv' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ta' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
処理完了：'datasetjson' に USUBJID が 01-704-1017 のデータを出力しました。
--- 試行回数: 1 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.132382
------
Event: node_started
Node: テキスト抽出ツール
------
Event: node_finished
Node: テキスト抽出ツール (succeeded)
Error: None
Elapsed time: 0.301828
------
Event: node_started
Node: IF/ELSE
------
Event: node_finished
Node: IF/ELSE (succeeded)
Error: None
Elapsed time: 0.142913
------
Event: node_started
Node: gemini-2.0-pro-exp
------
Event: node_finished
Node: gemini-2.0-pro-exp (failed)
Error: [google] Error: PluginInvokeError: {"args":null,"error

# 臨床試験データ統合レビュー報告

## 1. 症例サマリー：01-704-1017

*   **患者背景:** 77歳、男性、人種: WHITE、民族: NOT HISPANIC OR LATINO。
*   **イベント推移:**
    *   2013年09月20日 (Day -16): スクリーニング。教育レベル12年 (SC)。既往歴としてアルツハイマー病 (2011年診断)、心疾患関連（心筋梗塞 (2000年)、トリプルバイパス (2006年)など）あり (MH)。臨床検査にてクレアチニン高値 (HIGH, 159.12 umol/L, 基準範囲 71-141) (LB)。MMSEスコア21点、Hachinskiスコア0点 (QS)。
    *   2013年10月06日 (Day 1): ベースライン。治験薬 Xanomeline 54 mg/日 (PATCH, QD, TRANSDERMAL) 投与開始 (EX)。併用薬として PREMARIN (エストロゲン製剤) 0.625mg (QOD, ORAL) 開始 (CM)。立位血圧測定 (SYSBP/DIABP): 臥位 144/70 mmHg、立位1分後 120/66 mmHg、立位3分後 130/68 mmHg (VS)。
    *   2013年10月18日 (Day 13): Ambulatory ECG 装着。立位血圧測定 (SYSBP/DIABP): 臥位 134/64 mmHg、立位1分後 110/66 mmHg、立位3分後 116/70 mmHg (VS)。
    *   2013年10月19日 (Day 14): 有害事象「MYOCARDIAL INFARCTION」(MILD, 非重篤) 発現 (AE)。有害事象「VENTRICULAR SEPTAL DEFECT」(MILD, 非重篤) 発現 (AE)。有害事象「LATE EFFECTS OF CEREBRAL INFRACTION」 (SEVERE, 非重篤, AEDECOD: CARDIAC DISORDER) 発現 (AE)。臨床検査にて BUN高値 (HIGH, 10.353 mmol/L, 基準範囲 1.4-8.6)、Albumin低値 (LOW, 33 g/L, 基準範囲 35-46) (LB)。立位血圧測定 (SYSBP/DIABP): 臥位 112/60 mmHg、立位1分後 106/58 mmHg、立位3分後 104/56 mmHg (VS)。
    *   2013年10月20日 (Day 15): 治験薬 Xanomeline 81 mg/日へ増量 (EX)。
    *   2013年10月29日 (Day 24): 併用薬 PREMARIN 投与終了 (CM)。
    *   2013年11月01日 (Day 27): 臨床検査 (Week 4 Visit相当): Albumin低値 (LOW, 34 g/L, 基準範囲 35-46) (LB)。クレアチニン、BUNは基準範囲内。
    *   2013年11月05日 (Day 31): 有害事象「PRURITUS」(MILD, PROBABLE) 発現 (AE)。有害事象「RASH」(MILD, PROBABLE) 発現 (AE)。
    *   2013年11月06日 (Day 32): 併用薬 HYDROCORTISONE, TOPICAL 開始 (CM)。
    *   2013年11月09日 (Day 35): 立位血圧測定 (Week 4 Visit相当, SYSBP/DIABP): 臥位 124/66 mmHg、立位1分後 110/60 mmHg、立位3分後 106/60 mmHg (VS)。
    *   2013年11月18日 (Day 44): 治験薬 Xanomeline 投与終了 (EX)。有害事象「LATE EFFECTS OF CEREBRAL INFRACTION」 終了 (AE)。有害事象「BRAIN DEATH」(SEVERE, 非重篤) 発現および同日回復 (AE)。
    *   2013年11月19日 (Day 45): 有害事象「MYOCARDIAL INFARCTION」 回復 (AE)。(AEACN: DRUG WITHDRAWN)
    *   2013年11月22日 (Day 48): 併用薬 HYDROCORTISONE, TOPICAL 終了 (CM)。有害事象「PRURITUS」「RASH」 回復 (AEレコード AESEQ=8, 7) / 未回復 (AEレコード AESEQ=6, 5) の両方の記録あり。
    *   2013年11月24日 (Day 50): 有害事象により試験中止 (DS)。立位血圧測定 (Week 6 Visit相当, SYSBP/DIABP): 臥位 132/60 mmHg、立位1分後 114/60 mmHg、立位3分後 112/56 mmHg (VS)。NPI-Xにて幻覚 (Hallucinations) が新たに出現 (QS)。
    *   2013年12月06日 (Day 62): AEフォローアップ来院 (SV)。参加終了 (DM)。

## 2. 統合レビュー結果

*   **医学的観点からの指摘事項:**
    *   **指摘No.:** M-1
        *   **臨床試験結果/安全性への影響度合い:** Critical
        *   **内容:** 有害事象として「BRAIN DEATH」(AESEQ=3) が報告されているが、非重篤(AESER='N')と記録されている。Brain Death は定義上、死亡であり最も重篤なイベントである。また、同日に発現・回復(AESTDY=44, AEENDY=44) となっている点も医学的に極めて不自然。被験者は死亡しておらず(DM.DTHFL='', DM.DTHDTC='')、Day 50に試験中止(DS.DSSTDY=50)となっているため、誤報告の可能性が高い。
        *   **根拠:** AE.USUBJID='01-704-1017', AE.AESEQ=3, AE.AETERM='BRAIN DEATH', AE.AESER='N', AE.AESTDY=44, AE.AEENDY=44, AE.AEOUT='RECOVERED/RESOLVED'; DM.USUBJID='01-704-1017', DM.DTHFL='', DM.DTHDTC=''; DS.USUBJID='01-704-1017', DS.DSSTDY=50
    *   **指摘No.:** M-2
        *   **臨床試験結果/安全性への影響度合い:** Critical
        *   **内容:** 有害事象として「MYOCARDIAL INFARCTION」(AESEQ=1) が報告されているが、重症度 MILD、非重篤(AESER='N')と記録されている。心筋梗塞は通常、入院を要する、または生命を脅かす重篤なイベント (プロトコル Section 3.9.3.2.2 定義) に該当する可能性が高い。重症度・重篤度の評価が不適切である可能性がある。また、AEREL='NONE' となっているが、治験薬との関連を慎重に評価する必要がある (AEACN='DRUG WITHDRAWN' となっている点も考慮)。
        *   **根拠:** AE.USUBJID='01-704-1017', AE.AESEQ=1, AE.AETERM='MYOCARDIAL INFARCTION', AE.AESEV='MILD', AE.AESER='N', AE.AEREL='NONE', AE.AEACN='DRUG WITHDRAWN', AE.AESTDY=14, AE.AEENDY=45; プロトコル Section 3.9.3.2.2
    *   **指摘No.:** M-3
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **内容:** 有害事象として「VENTRICULAR SEPTAL DEFECT」(AESEQ=2) が報告されている。これは通常、先天性疾患であり、有害事象ではなく既往歴(MH)として扱われるべきである。AESTDY=14 となっており、治験期間中に新たに診断された可能性もあるが、AEとしての報告の妥当性を確認する必要がある。
        *   **根拠:** AE.USUBJID='01-704-1017', AE.AESEQ=2, AE.AETERM='VENTRICULAR SEPTAL DEFECT', AE.AEDECOD='VENTRICULAR SEPTAL DEFECT', AE.AEBODSYS='CONGENITAL, FAMILIAL AND GENETIC DISORDERS', AE.AESTDY=14
    *   **指摘No.:** M-4
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **内容:** 有害事象として「LATE EFFECTS OF CEREBRAL INFRACTION」(AESEQ=4) が報告されている。これは後遺症であり、通常、有害事象ではなく既往歴(MH)に関連する状態と考えられる。AETERMとAEDECOD ('CARDIAC DISORDER') の不一致も見られ、報告内容の妥当性が疑われる。
        *   **根拠:** AE.USUBJID='01-704-1017', AE.AESEQ=4, AE.AETERM='LATE EFFECTS OF CEREBRAL INFRACTION', AE.AEDECOD='CARDIAC DISORDER', AE.AESTDY=14, AE.AEENDY=44
    *   **指摘No.:** M-5
        *   **臨床試験結果/安全性への影響度合い:** Minor
        *   **内容:** 臨床検査にて、ベースラインのクレアチニン(CREAT)高値、Day 14のBUN高値、Day 14およびDay 27のアルブミン(ALB)低値が認められている。これらの異常値の臨床的意義と治験薬との関連について評価が必要。特にベースラインのクレアチニン高値は選択除外基準との関連で確認が必要。
        *   **根拠:** LB.USUBJID='01-704-1017', LB.LBTESTCD='CREAT', LB.LBDY=-16, LB.LBNRIND='HIGH'; LB.LBTESTCD='BUN', LB.LBDY=14, LB.LBNRIND='HIGH'; LB.LBTESTCD='ALB', LB.LBDY=14, LB.LBNRIND='LOW'; LB.LBTESTCD='ALB', LB.LBDY=27, LB.LBNRIND='LOW'
    *   **指摘No.:** M-6
        *   **臨床試験結果/安全性への影響度合い:** Minor
        *   **内容:** バイタルサインにおいて、特に治験薬増量後の Day 14, Day 35, Day 50 の立位時に収縮期および拡張期血圧の低下傾向が見られる (例: Day 14 立位3分後 SYSBP 104 mmHg / DIABP 56 mmHg)。起立性低血圧の可能性を考慮し、臨床症状との関連を確認する必要がある。
        *   **根拠:** VS.USUBJID='01-704-1017', VS.VSDY=1, 13, 14, 35, 50 における VS.VSTESTCD='SYSBP'/'DIABP' および VS.VSPOS='SUPINE'/'STANDING' の値。

*   **データ整合性観点からの指摘事項:**
    *   **指摘No.:** D-1
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **内容:** AEドメインにおいて、AESEQ=2 (VENTRICULAR SEPTAL DEFECT) の終了日(AEENDTC)が欠損している。また、AESEQ=6, 5 (PRURITUS, RASH) と AESEQ=8, 7 (PRURITUS, RASH) で、同じ事象に対して異なる転帰 (NOT RECOVERED/NOT RESOLVED と RECOVERED/RESOLVED) および異なる収集日(AEDTC)が記録されており、重複または矛盾した記録となっている。RELRECドメインでは、AESEQ 5と7が同一のRELID ('01-704-1017-E11') で関連付けられているが、AESEQ 6と8の関連付けがない。
        *   **根拠:** AE.USUBJID='01-704-1017', AE.AESEQ=2, AE.AEENDTC is null; AE.USUBJID='01-704-1017', AE.AETERM='PRURITUS' (AESEQ=6 vs 8), AE.AETERM='RASH' (AESEQ=5 vs 7); RELREC.USUBJID='01-704-1017', RELREC.RELID='01-704-1017-E11'
    *   **指摘No.:** D-2
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **内容:** Visit Number と Study Day (--DY) および日付 (--DTC) の整合性に問題がある。特に Visit 5 (Week 4, Planned Day 28) では、SVドメインの訪問日は2013-11-09 (VISITDY=28) だが、LB測定は2013-11-01 (LBDY=27)、VS/QS測定は2013-11-09 (VSDY/QSDY=35) となっている。Visit 7 (Week 6, Planned Day 42) でも、SV訪問日は2013-11-24 (VISITDY=42) だが、VS/QS測定日は2013-11-24 (VSDY/QSDY=50) となっている。同一Visit内で測定日やStudy Dayが異なる、またはPlanned Dayと実際のStudy Dayが大きく乖離している。
        *   **根拠:** SV, LB, VS, QS ドメインの VISITNUM, VISITDY, --DTC, --DY の比較。例: VISITNUM=5, 7。
    *   **指摘No.:** D-3
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **内容:** 重要なデータに欠損が見られる。AEドメインでは、複数のイベントで関連性(AEREL)や処置(AEACN)が空欄になっている。また、プロトコルで Week 6 以降も予定されている臨床検査 (LB) のデータが Visit 5 (Day 27) 以降欠損している。
        *   **根拠:** AEドメインの AEREL, AEACN が "" のレコード (例: AESEQ=3, 4, 5, 6, 7, 8)。LBドメインのデータが LBDY=27 (VISITNUM=5) までしかない。プロトコル Attachment LZZT.1。
    *   **指摘No.:** D-4
        *   **臨床試験結果/安全性への影響度合い:** Minor
        *   **内容:** CMドメインの HYDROCORTISONE, TOPICAL (CMSEQ=9, 11, 13) において、CMDECOD および CMCLAS が 'UNCODED' となっている。標準化された薬剤名や分類が割り当てられていない。
        *   **根拠:** CM.USUBJID='01-704-1017', CM.CMTRT='HYDROCORTISONE, TOPICAL', CM.CMDECOD='UNCODED', CM.CMCLAS='UNCODED'

*   **プロトコル遵守観点からの指摘事項 (逸脱の可能性):**
    *   **指摘No.:** P-1
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **内容:** ベースラインのクレアチニン値が基準範囲上限を超えていた (LB.LBNRIND='HIGH')。除外基準 EXCL27b では「Laboratory test values exceeding the Lilly Reference Range III for the patient’s age in any of the following analytes: creatinine...」と規定されている。Lilly Reference Range III for 77歳男性 の具体的な値は不明だが、基準値超過での登録はプロトコル逸脱の可能性がある。また、併用薬 PREMARIN が Day 1 から開始されている。除外基準 EXCL31v では「Estrogen supplements are permitted during the study, but dosage must be stable for at least 3 months prior to enrollment.」と規定されており、治験開始後の新規投与開始は逸脱の可能性がある。
        *   **プロトコル該当箇所:** Section 3.4.2.2 Exclusion Criteria [27b], [31v]
        *   **根拠:** LB.USUBJID='01-704-1017', LB.LBTESTCD='CREAT', LB.LBDY=-16, LB.LBNRIND='HIGH'; CM.USUBJID='01-704-1017', CM.CMTRT='PREMARIN', CM.CMSTDTC='2013-10-06' (Day 1)
    *   **指摘No.:** P-2
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **内容:** High Dose群の治験薬投与量について、プロトコル Figure LZZT.1 では最初の8週間は54mg(50cm2)を投与し、その後81mg(75cm2)に増量する計画が示唆されているが、本症例ではDay 15 (Week 2終了時) に54mgから81mgへ増量されている。プロトコル本文 Section 3.1 には開始用量のみ記載があり増量時期は明記されていないが、図との不整合があり、計画からの逸脱の可能性がある。
        *   **プロトコル該当箇所:** Section 3.1 Summary of Study Design, Figure LZZT.1
        *   **根拠:** EX.USUBJID='01-704-1017', EX.EXDOSE=54, EX.EXSTDY=1, EX.EXENDY=14; EX.EXDOSE=81, EX.EXSTDY=15, EX.EXENDY=44
    *   **指摘No.:** P-3
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **内容:** 評価スケジュールからの逸脱。Visit 5 (Week 4), Visit 7 (Week 6) などで、VS, LB, QS の測定日が計画日(VISITDY)からずれている (D-2参照)。また、プロトコルでは ADAS-Cog, CIBIC+, DAD は Visit 3, 8, 10, 12 で実施予定だが、データ上では Visit 4, 5, 7 で実施されている。NPI-X もプロトコル上の2週間毎の評価スケジュールと実際の測定 Visit/Day が一致しない可能性がある (データは Visit 3, 4, 5, 7 で記録)。
        *   **プロトコル該当箇所:** Section 3.9.1.1 Efficacy Measures, Attachment LZZT.1 Schedule of Events
        *   **根拠:** SV, LB, VS, QS ドメインの VISITNUM, VISITDY, --DTC, --DY の比較 (D-2参照); QSドメインの VISITNUM (3, 4, 5, 7) とプロトコルの実施予定 Visit (3, 8, 10, 12) の比較。
    *   **指摘No.:** P-4
        *   **臨床試験結果/安全性への影響度合い:** Critical
        *   **内容:** 重篤な有害事象 (Serious Adverse Event: SAE) の報告基準逸脱の可能性。「BRAIN DEATH」および「MYOCARDIAL INFARCTION」が非重篤 (AESER='N') と報告されているが、プロトコル Section 3.9.3.2.2 のSAE定義（死亡、生命を脅かす、入院または入院期間の延長など）に該当する可能性が高い。SAEとして適切に評価・報告されていない疑いがある。
        *   **プロトコル該当箇所:** Section 3.9.3.2.2 Serious Adverse Events
        *   **根拠:** AE.USUBJID='01-704-1017', AE.AESEQ=3, AE.AETERM='BRAIN DEATH', AE.AESER='N'; AE.AESEQ=1, AE.AETERM='MYOCARDIAL INFARCTION', AE.AESER='N'

## 3. 疑義事項

*   **医療機関へのクエリ:**
    *   **クエリNo.:** Q-1 (関連指摘No.: M-1, D-1, P-4)
        *   **臨床試験結果/安全性への影響度合い:** Critical
        *   **医療機関への問い合わせ文面:** 有害事象として Day 44 に発現・回復した「BRAIN DEATH」(AESEQ=3) が報告されていますが、非重篤(AESER='N')と記録されています。イベントの詳細（実際の事象、診断根拠）、重篤性評価（SAE基準との照合）、転帰について再確認し、必要に応じて修正してください。
        *   **判断理由:** 報告された事象名と重篤度評価、転帰に重大な矛盾があり、データの正確性と患者安全性の観点から確認が必須なため。
        *   **判断根拠:**
            *   AE.USUBJID = '01-704-1017', AE.AESEQ = 3, AE.AETERM = 'BRAIN DEATH', AE.AESER = 'N', AE.AESTDY = 44, AE.AEENDY = 44, AE.AEOUT = 'RECOVERED/RESOLVED'
            *   DM.USUBJID = '01-704-1017', DM.DTHFL = '', DM.DTHDTC = ''
            *   プロトコル Section 3.9.3.2.2 (SAE定義)
    *   **クエリNo.:** Q-2 (関連指摘No.: M-2, P-4)
        *   **臨床試験結果/安全性への影響度合い:** Critical
        *   **医療機関への問い合わせ文面:** 有害事象「MYOCARDIAL INFARCTION」(AESEQ=1) が Day 14 に発現し、重症度 MILD、非重篤(AESER='N')と報告されています。心筋梗塞は通常、重篤なイベントに該当する可能性があります。SAE基準に基づき、重篤性評価（入院の有無、生命を脅かす状態であったか等）を再確認してください。また、治験薬との関連性(AEREL='NONE')についても再評価をお願いします (AEACN='DRUG WITHDRAWN' となっています)。
        *   **判断理由:** SAE評価の妥当性、および治験薬との関連性評価の確認が必要なため。
        *   **判断根拠:**
            *   AE.USUBJID = '01-704-1017', AE.AESEQ = 1, AE.AETERM = 'MYOCARDIAL INFARCTION', AE.AESEV = 'MILD', AE.AESER = 'N', AE.AEREL = 'NONE', AE.AEACN = 'DRUG WITHDRAWN', AE.AESTDY = 14, AE.AEENDY = 45
            *   プロトコル Section 3.9.3.2.2 (SAE定義)
    *   **クエリNo.:** Q-3 (関連指摘No.: M-3, D-1)
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 有害事象として Day 14 に発現した「VENTRICULAR SEPTAL DEFECT」(AESEQ=2) が報告されています。これは通常先天性の状態ですが、AEとしての報告で正しいでしょうか？ もしAEとして正しい場合、終了日(AEENDTC)と転帰(AEOUT)が未入力です。ご確認ください。もし既往歴であればMHドメインへの記録をご検討ください。
        *   **判断理由:** 事象の分類（AE vs MH）の妥当性、およびAE記録の完全性を確認するため。
        *   **判断根拠:**
            *   AE.USUBJID = '01-704-1017', AE.AESEQ = 2, AE.AETERM = 'VENTRICULAR SEPTAL DEFECT', AE.AEBODSYS = 'CONGENITAL, FAMILIAL AND GENETIC DISORDERS', AE.AESTDY = 14, AE.AEENDTC = null, AE.AEOUT is null
    *   **クエリNo.:** Q-4 (関連指摘No.: M-4)
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 有害事象として「LATE EFFECTS OF CEREBRAL INFRACTION」(AESEQ=4) が報告されていますが、事象名(AETERM)と MedDRAコード化された Body System (AEDECOD='CARDIAC DISORDER') が一致していません。また、後遺症がAEとして報告されるのは一般的ではありません。報告内容（実際の事象、診断根拠、AEとしての妥当性）をご確認ください。
        *   **判断理由:** 報告されたイベント内容の正確性と妥当性を確認するため。
        *   **判断根拠:**
            *   AE.USUBJID = '01-704-1017', AE.AESEQ = 4, AE.AETERM = 'LATE EFFECTS OF CEREBRAL INFRACTION', AE.AEDECOD = 'CARDIAC DISORDER', AE.AESTDY = 14, AE.AEENDY = 44
    *   **クエリNo.:** Q-5 (関連指摘No.: P-1)
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** スクリーニング時(Day -16)のクレアチニン値が 159.12 umol/L で基準範囲(71-141 umol/L)を超えていますが、被験者は登録されています。除外基準 [27b] (基準範囲を超える検査値) との関連について、登録可能と判断された理由をご教示ください。
        *   **判断理由:** 選択/除外基準の遵守状況を確認するため。
        *   **判断根拠:**
            *   LB.USUBJID = '01-704-1017', LB.LBTESTCD = 'CREAT', LB.LBDY = -16, LB.LBSTRESN = 159.12, LB.LBSTNRHI = 141, LB.LBNRIND = 'HIGH'
            *   プロトコル Section 3.4.2.2 Exclusion Criteria [27b]
    *   **クエリNo.:** Q-6 (関連指摘No.: P-1)
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** 併用薬「PREMARIN」(Estrogens conjugated) が Day 1 から Day 24 まで投与されています。除外基準 [31v] では、エストロゲン補充療法は登録前3ヶ月間用量が安定している場合にのみ許可されています。治験開始後の新規投与開始について、プロトコル遵守の観点から理由と経緯をご確認ください。
        *   **判断理由:** 除外基準および併用薬規定の遵守状況を確認するため。
        *   **判断根拠:**
            *   CM.USUBJID = '01-704-1017', CM.CMTRT = 'PREMARIN', CM.CMSTDTC = '2013-10-06', CM.CMENDTC = '2013-10-29'
            *   プロトコル Section 3.4.2.2 Exclusion Criteria [31v]
    *   **クエリNo.:** Q-7 (関連指摘No.: D-2, P-3)
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** Visit 5 (Week 4, 計画 Day 28) および Visit 7 (Week 6, 計画 Day 42) において、検査・評価の実施日が計画日からずれています (例: Visit 5 の VS/QS は Day 35、Visit 7 の VS/QS は Day 50)。また、Visit 5 では LB の測定日(Day 27) と VS/QS の測定日(Day 35)が異なります。評価スケジュールのずれについて理由をご確認ください。
        *   **判断理由:** プロトコルで規定された評価スケジュールの遵守状況とデータの正確性を確認するため。
        *   **判断根拠:**
            *   SV.VISITNUM=5, SV.VISITDY=28, SV.SVSTDTC='2013-11-09'; LB.VISITNUM=5, LB.LBDY=27, LB.LBDTC='2013-11-01'; VS.VISITNUM=5, VS.VSDY=35, VS.VSDTC='2013-11-09'; QS.VISITNUM=5, QS.QSDY=35, QS.QSDTC='2013-11-09'
            *   SV.VISITNUM=7, SV.VISITDY=42, SV.SVSTDTC='2013-11-24'; VS.VISITNUM=7, VS.VSDY=50, VS.VSDTC='2013-11-24'; QS.VISITNUM=7, QS.QSDY=50, QS.QSDTC='2013-11-24'
            *   プロトコル Attachment LZZT.1 Schedule of Events
    *   **クエリNo.:** Q-8 (関連指摘No.: P-3)
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **医療機関への問い合わせ文面:** ADAS-Cog, CIBIC+, DAD が、プロトコルで規定された Visit (Visit 8, 10, 12) ではなく、Visit 4, 5, 7 で実施されています。評価スケジュールの変更があったのでしょうか、あるいは記録誤りでしょうか。ご確認ください。
        *   **判断理由:** 主要評価項目の評価スケジュールの遵守状況を確認するため。
        *   **判断根拠:**
            *   QS.USUBJID = '01-704-1017', QS.QSCAT = 'ALZHEIMER''S DISEASE ASSESSMENT SCALE'/'CLINICIAN''S INTERVIEW-BASED IMPRESSION OF CHANGE (CIBIC+)'/'DISABILITY ASSESSMENT FOR DEMENTIA (DAD)', QS.VISITNUM = 3, 7 (データ上は Week 6 の Visit 7 で実施。Visit 8, 10, 12 のデータなし)
            *   プロトコル Section 3.9.1.1, Attachment LZZT.1 Schedule of Events

*   **内部確認事項 (問い合わせ不要):**
    *   **確認事項No.:** I-1 (関連指摘No.: P-2)
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **疑義事項/確認内容:** High Dose群の治験薬増量タイミングが Day 15 となっており、プロトコル Figure LZZT.1 で示唆される 8週目からの増量と異なる。プロトコル本文には増量時期の明確な記載がないため、Figure LZZT.1 の解釈を含め、社内で投与計画の意図を再確認する。
        *   **判断理由:** 投与計画の解釈に関する内部確認であり、医療機関への問い合わせは現時点で不要と判断。
        *   **判断根拠:**
            *   EX.USUBJID = '01-704-1017', EX.EXSTDY = 1, EX.EXENDY = 14, EX.EXDOSE = 54; EX.EXSTDY = 15, EX.EXENDY = 44, EX.EXDOSE = 81
            *   プロトコル Section 3.1 Summary of Study Design, Figure LZZT.1
    *   **確認事項No.:** I-2 (関連指摘No.: D-1)
        *   **臨床試験結果/安全性への影響度合い:** Minor
        *   **疑義事項/確認内容:** AEドメインの PRURITUS (AESEQ=6, 8) および RASH (AESEQ=5, 7) について、同一事象に対する重複・矛盾した記録が存在する。RELRECでの関連付けも不完全。データクリーニングプロセスにおいて修正・統合が必要か検討する。
        *   **判断理由:** データマネジメント上の問題であり、内部での対応が可能と判断。
        *   **判断根拠:**
            *   AE.USUBJID = '01-704-1017', AE.AETERM = 'PRURITUS' (AESEQ=6 vs 8), AE.AETERM = 'RASH' (AESEQ=5 vs 7)
            *   RELREC.USUBJID = '01-704-1017', RELREC.RELID = '01-704-1017-E11'
    *   **確認事項No.:** I-3 (関連指摘No.: D-3)
        *   **臨床試験結果/安全性への影響度合い:** Minor
        *   **疑義事項/確認内容:** AEドメインにおいて、関連性(AEREL)や処置(AEACN)が空欄 ("") となっているレコードが複数存在する。入力規則として許容されているか、あるいは入力漏れかを確認する。
        *   **判断理由:** データ入力規則や標準的な運用に関する内部確認事項。
        *   **判断根拠:**
            *   AE.USUBJID = '01-704-1017', AE.AEREL="", AE.AEACN="" のレコード (例: AESEQ=3, 4, 5, 6, 7, 8)
    *   **確認事項No.:** I-4 (関連指摘No.: D-3)
        *   **臨床試験結果/安全性への影響度合い:** Major
        *   **疑義事項/確認内容:** Week 6以降に予定されていた臨床検査 (LB) のデータが存在しない。被験者は Day 50 (Week 6 Visit相当) に試験中止となっているため、中止前に採取されなかった可能性があるが、データ欠損の理由を明確にする必要がある。
        *   **判断理由:** データ欠損の理由特定は内部調査で可能な場合があるため。
        *   **判断根拠:**
            *   LBドメインの最新データが LBDY=27 (VISITNUM=5)
            *   DS.USUBJID = '01-704-1017', DS.DSSTDY = 50
            *   プロトコル Attachment LZZT.1 Schedule of Events
    *   **確認事項No.:** I-5 (関連指摘No.: D-4)
        *   **臨床試験結果/安全性への影響度合い:** Minor
        *   **疑義事項/確認内容:** CMドメインの HYDROCORTISONE, TOPICAL について、CMDECOD および CMCLAS が 'UNCODED' となっている。標準辞書でのコーディング状況を確認し、必要に応じて修正する。
        *   **判断理由:** データ標準化に関する内部確認事項。
        *   **判断根拠:**
            *   CM.USUBJID = '01-704-1017', CM.CMTRT = 'HYDROCORTISONE, TOPICAL', CM.CMDECOD = 'UNCODED', CM.CMCLAS = 'UNCODED'

,Subject,Task_single
0,01-704-1017,# 臨床試験データ統合レビュー報告\n\n## 1. 症例サマリー：01-704-1017\...


## 結果の出力

### JSONの場合の出力関数

In [ ]:
import re
import json

def extract_json(text):
    # ```json ... ``` のパターンを検索 (非貪欲マッチ)
    match = re.search(r'```json\s*([\s\S]*?)\s*```', text)

    if match:
        json_string = match.group(1)
        try:
            data = json.loads(json_string)
            return data
        except json.JSONDecodeError:
            print("Error: Invalid JSON found.")
            return None
    else:
        print("Error: No JSON code block found.")
        return None

# Test
#extract_json(df_results['Task1'][0])


In [ ]:
import json
from datetime import datetime

# --- 新しいヘルパー関数 ---
def format_partial_date(date_str):
    """
    部分日付を含む日付文字列を可能な限り指定の日本語形式にフォーマットする。
    対応形式: YYYY-MM-DD, YYYY-MM, YYYY
    """
    if not date_str:
        return '日付不明'

    # 優先度順にフォーマットを試す
    formats_map = {
        '%Y-%m-%d': '%Y年%m月%d日',
        '%Y-%m': '%Y年%m月',
        '%Y': '%Y年'
    }

    for input_format, output_format in formats_map.items():
        try:
            date_obj = datetime.strptime(date_str, input_format)
            return date_obj.strftime(output_format)
        except ValueError:
            continue # 次のフォーマットを試す

    # どの形式にも一致しない場合は、元の文字列をそのまま返す
    # (予期しない形式や "UNKNOWN" などの文字列に対応するため)
    return date_str
# --- ヘルパー関数ここまで ---

def format_variables_list(variables):
    """
    変数リストをMarkdownの箇条書き形式の複数行文字列にフォーマットする関数
    各行は '* 変数名 = 値' の形式
    """
    if not variables:
        return [] # 空のリストを返す
    lines = [f"* {v.get('variable', 'N/A')} = {v.get('value', 'N/A')}" for v in variables]
    return lines

def indent_lines(lines, indent_spaces):
    """指定された行リストの各行にインデントを追加する"""
    indent = " " * indent_spaces
    return "\n".join([indent + line for line in lines])

def generate_queries_section(queries, section_title="医療機関に問い合わせるクエリ", no_label="クエリNo."):
    """
    クエリセクションのMarkdownを生成する関数 (判断根拠を箇条書き表示)
    """
    markdown = []
    has_queries = bool(queries) # None や空リストでないかチェック

    markdown.append(f"{section_title}: {'あり' if has_queries else 'なし'}")
    if has_queries:
        for query in queries:
            markdown.append(f"    *   **{no_label}:** {query.get('query_no', 'N/A')}")
            markdown.append(f"        *   **臨床試験結果への影響度合い:** {query.get('criticality', 'N/A')}")
            markdown.append(f"        *   **医療機関への問い合わせ文面:** {query.get('inquiry', 'N/A')}")
            markdown.append(f"        *   **判断理由:** {query.get('reason', 'N/A')}")
            markdown.append(f"        *   **判断根拠:**")
            variable_lines = format_variables_list(query.get('variables', []))
            if variable_lines:
                markdown.append(indent_lines(variable_lines, 12))

    return "\n".join(markdown)

# --- generate_timeline_markdown を修正 ---
def generate_timeline_markdown(data):
    """
    'timeline' キーが存在する場合のMarkdownを生成する関数 (部分日付対応)
    """
    usubjid = data.get('usubjid', 'N/A')
    timeline = data.get('timeline', [])
    queries = data.get('queries')

    markdown = []
    markdown.append(f"1. 症例サマリー：{usubjid}")

    for entry in timeline:
        # 新しいヘルパー関数を使って日付をフォーマット
        formatted_date = format_partial_date(entry.get('date'))

        day = entry.get('day', '不明') # Day は日付形式に関わらず表示
        details = entry.get('details', '詳細不明')
        markdown.append(f"    *   {formatted_date} (Day {day}): {details}")

    markdown.append("") # 空行
    markdown.append(generate_queries_section(queries, section_title="2. 疑義事項", no_label="クエリNo."))

    return "\n".join(markdown)
# --- generate_timeline_markdown の修正ここまで ---

def generate_data_issues_markdown(data):
    """
    'data_issues' キーが存在する場合のMarkdownを生成する関数 (判断根拠を箇条書き表示)
    """
    usubjid = data.get('usubjid', 'N/A')
    data_issues = data.get('data_issues', [])
    queries = data.get('queries')

    markdown = []
    markdown.append(f"1. 確認した症例：{usubjid}")
    markdown.append("") # 空行

    markdown.append(generate_queries_section(queries, section_title="2. 医療機関に問い合わせるクエリ", no_label="クエリNo."))
    markdown.append("") # 空行

    has_data_issues = bool(data_issues)
    markdown.append(f"3. 医療機関に問い合わせない疑義事項: {'あり' if has_data_issues else 'なし'}")
    if has_data_issues:
        for issue in data_issues:
            markdown.append(f"    *   **疑義No.:** {issue.get('issue_no', 'N/A')}")
            markdown.append(f"        *   **疑義事項:** {issue.get('inconsistency', 'N/A')}")
            markdown.append(f"        *   **判断理由:** {issue.get('cause', 'N/A')}")
            markdown.append(f"        *   **判断根拠:")
            variable_lines = format_variables_list(issue.get('variables', []))
            if variable_lines:
                markdown.append(indent_lines(variable_lines, 12))

    return "\n".join(markdown)

def generate_deviations_markdown(data):
    """
    'deviations' キーが存在する場合のMarkdownを生成する関数 (判断根拠を箇条書き表示)
    """
    usubjid = data.get('usubjid', 'N/A')
    deviations = data.get('deviations', [])
    queries = data.get('queries')

    markdown = []
    markdown.append(f"1. 確認した症例：{usubjid}")
    markdown.append("") # 空行

    has_deviations = bool(deviations)
    markdown.append(f"2. プロトコル逸脱: {'あり' if has_deviations else 'なし'}")
    if has_deviations:
        for deviation in deviations:
            markdown.append(f"    *   **逸脱No.:** {deviation.get('deviation_no', 'N/A')}")
            markdown.append(f"        *   **臨床試験結果への影響度合い:** {deviation.get('impact', 'N/A')}")
            markdown.append(f"        *   **逸脱内容:** {deviation.get('description', 'N/A')}")
            markdown.append(f"        *   **プロトコル該当箇所:** {deviation.get('protocol_reference', 'N/A')}")
            markdown.append(f"        *   **判断理由:** {deviation.get('justification', 'N/A')}")
            markdown.append(f"        *   **判断根拠:")
            variable_lines = format_variables_list(deviation.get('variables', []))
            if variable_lines:
                markdown.append(indent_lines(variable_lines, 12))
    markdown.append("") # 空行

    markdown.append(generate_queries_section(queries, section_title="3. 医療機関に問い合わせるクエリ", no_label="クエリNo."))

    return "\n".join(markdown)

def json_to_markdown(json_input):
    """
    JSONデータを受け取り、指定の形式のMarkdownに変換するメイン関数 (部分日付対応)
    """
    try:
        if isinstance(json_input, str):
            data = json.loads(json_input)
        elif isinstance(json_input, dict):
            data = json_input
        else:
            return "エラー: 入力はJSON文字列またはPython辞書である必要があります。"
    except json.JSONDecodeError:
        return "エラー: 無効なJSON文字列です。"
    except Exception as e:
        return f"エラー: 予期せぬエラーが発生しました - {e}"

    if 'timeline' in data:
        return generate_timeline_markdown(data)
    elif 'data_issues' in data:
        return generate_data_issues_markdown(data)
    elif 'deviations' in data:
        return generate_deviations_markdown(data)
    else:
        usubjid = data.get('usubjid', 'N/A')
        queries = data.get('queries')
        markdown = []
        markdown.append(f"1. 確認した症例：{usubjid}")
        markdown.append("\n---\n")
        markdown.append("入力データには timeline, data_issues, deviations のいずれのキーも含まれていません。")
        if queries is not None:
             markdown.append("\n---\n")
             markdown.append(generate_queries_section(queries, section_title="クエリ情報", no_label="クエリNo."))
        return "\n".join(markdown)

# Test
#markdown_output = json_to_markdown(extract_json(df_results['Task3'][0]))
#print(markdown_output)


In [ ]:
def result_output_JSON(df: pd.DataFrame) -> str:
    """
    DataFrameを指定されたテキスト形式に変換します。

    Args:
        df: 変換するDataFrame。カラム名は 'Subject', 'Task1', 'Task2', 'Task3' である必要があります。

    Returns:
        変換後のテキストデータ。
    """
    text_data = ""
    for index, row in df.iterrows():
        subject = row['Subject']
        task1 = json_to_markdown(extract_json(row['Task1']))
        task2 = json_to_markdown(extract_json(row['Task2']))
        task3 = json_to_markdown(extract_json(row['Task3']))

        text_data += f"# {subject}\n"
        text_data += f"## Task1: Clinical Review Results\n"
        text_data += f"{task1}\n\n"
        text_data += f"## Task2: DM Review Results\n"
        text_data += f"{task2}\n\n"
        text_data += f"## Task3: Protocol Deviation Review Results\n"
        text_data += f"{task3}\n\n"

    return text_data

#output_text = result_output_JSON(df_results)
#print(output_text)

### Markdownの場合の出力関数

In [ ]:

def result_output_Markdown(df: pd.DataFrame) -> str:
    """
    DataFrameを指定されたテキスト形式に変換します。

    Args:
        df: 変換するDataFrame。カラム名は 'Subject', 'Task1', 'Task2', 'Task3' である必要があります。

    Returns:
        変換後のテキストデータ。
    """
    text_data = ""
    for index, row in df.iterrows():
        subject = row['Subject']
        task1 = row['Task1']
        task2 = row['Task2']
        task3 = row['Task3']

        text_data += f"# {subject}\n"
        text_data += f"## Task1: Clinical Review Results\n"
        text_data += f"{task1}\n"
        text_data += f"## Task2: DM Review Results\n"
        text_data += f"{task2}\n"
        text_data += f"## Task3: Protocol Deviation Review Results\n"
        text_data += f"{task3}\n\n"

    return text_data

#output_text = result_output_Markdown(df_results)

### 出力

In [ ]:
if Output_Format == "Markdown":
  output_text = result_output_Markdown(df_results)
elif Output_Format == "JSON":
  output_text = result_output_JSON(df_results)

# mdファイルに保存
output_file = 'output_' + ModelName + '.md'  # 保存するファイル名を指定
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(output_text)

print(output_text)

# 01-701-1015
## Task1: Clinical Review Results
1. 症例サマリー：[01-701-1015]
    *   被験者は63歳女性、アルツハイマー病の診断 (2010年4月30日)。プラセボ群に割り付けられた。
    *   **スクリーニング期間 (Day -7 ～ Day -2):**
        *   2013年12月26日 (Day -7): 臨床検査にてALP低値 (34 U/L, 基準値 35-115)、AST高値 (40 U/L, 基準値 9-34)、赤血球大小不同 (Anisocytes) 異常 (1) を認めた。収縮期血圧は立位3分後に147 mmHgとやや高値を示した。
        *   2013年12月31日 (Day -2): 収縮期血圧は立位3分後に145 mmHgとやや高値を示した。
    *   **治験薬投与期間 (Day 1 ～ Day 182):**
        *   2014年01月02日 (Day 1): プラセボ投与開始。ベースラインの臥位収縮期血圧は130 mmHg。
        *   2014年01月03日 (Day 2): 投与部位紅斑 (APPLICATION SITE ERYTHEMA, MILD) および投与部位掻痒感 (APPLICATION SITE PRURITUS, MILD) が発現。両事象とも治験終了時まで未回復。同日より、併用薬としてNEOSPORIN (外用) の使用を開始 (PRN)。
        *   2014年01月11日 (Day 10): 下痢 (DIARRHOEA, MILD) が発現。本有害事象は重篤 (SAE) と判断され、入院を要した。治験薬との関連性は低い (REMOTE) と評価された。**（開始日 2014-01-11 に対し、終了日が 2014-01-09 と記録されており、日付に矛盾あり）**
        *   2014年01月16日 (Day 15): 臨床検査にてALT高値 (41 U/L, 基準値 6-34) を認めたが、次回以降は正常範囲内に回復。
        *   2014年01月30日 (Day 29): 臨床検査にてMCV低値 (78 fL, 基準値 80-100

# 以下メモ

In [ ]:
# Schema memo
'''
{
  "name": "Clinical_Review",
  "description": "Schema for clinical case summaries and associated queries",
  "strict": true,
  "schema": {
    "type": "object",
    "properties": {
      "usubjid": {
        "type": "string",
        "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)"
      },
      "timeline": {
        "type": ["array", "null"],
        "description": "Chronological events in the subject's case",
        "items": {
          "type": "object",
          "properties": {
            "date": {
              "type": "string",
              "description": "Date of the event.  Allows full (YYYY-MM-DD), partial (YYYY-MM), or year-only (YYYY) formats."
            },
            "day": {
              "type": "integer",
              "description": "Day relative to study start (Day 1).  Allows negative values for pre-treatment days."
            },
            "details": {
              "type": "string",
              "description": "Detailed information about the event (e.g., adverse event, medication administration)"
            }
          },
          "required": [
            "date",
            "day",
            "details"
          ],
          "additionalProperties": false
        }
      },
      "data_issues": {
        "type": ["array", "null"],
        "description": "List of data issues identified during review",
        "items": {
          "type": "object",
          "properties": {
            "issue_no": {
              "type": "integer",
              "description": "Unique issue number"
            },
            "variables": {
              "type": "array",
              "description": "List of variables and their values related to the issue",
              "items": {
                "type": "object",
                "properties": {
                  "variable": {
                    "type": "string",
                    "description": "Variable name"
                  },
                  "value": {
                    "type": "string",
                    "description": "Value of the variable"
                  }
                },
                "required": [
                  "variable",
                  "value"
                ],
                "additionalProperties": false
              }
            },
            "inconsistency": {
              "type": "string",
              "description": "Description of the data inconsistency"
            },
            "cause": {
              "type": "string",
              "description": "Suspected cause of the issue"
            },
            "resolution": {
              "type": "string",
              "description": "Proposed resolution for the issue"
            }
          },
          "required": [
            "issue_no",
            "variables",
            "inconsistency",
            "cause",
            "resolution"
          ],
          "additionalProperties": false
        }
      },
      "deviations": {
        "type": ["array", "null"],
        "description": "List of protocol deviations",
        "items": {
          "type": "object",
          "properties": {
            "deviation_no": {
              "type": "integer",
              "description": "Unique deviation number"
            },
            "impact": {
              "type": "string",
              "description": "Impact of the deviation on the clinical trial results",
              "enum": ["Critical", "Major", "Minor"]
            },
            "variables": {
              "type": "array",
              "description": "List of variables and their values related to the deviation",
              "items": {
                "type": "object",
                "properties": {
                  "variable": {
                    "type": "string",
                    "description": "Variable name"
                  },
                  "value": {
                    "type": "string",
                    "description": "Value of the variable"
                  }
                },
                "required": [
                  "variable",
                  "value"
                ],
                "additionalProperties": false
              }
            },
            "description": {
              "type": "string",
              "description": "Description of the deviation"
            },
            "protocol_reference": {
              "type": "string",
              "description": "Reference to the relevant section in the protocol"
            },
            "justification": {
              "type": "string",
              "description": "Justification for the deviation classification"
            }
          },
          "required": [
            "deviation_no",
            "impact",
            "variables",
            "description",
            "protocol_reference",
            "justification"
          ],
          "additionalProperties": false
        }
      },
      "queries": {
        "type": ["array", "null"],
        "description": "List of queries related to the subject",
        "items": {
          "type": "object",
          "properties": {
            "query_no": {
              "type": "integer",
              "description": "Unique query number"
            },
            "criticality": {
              "type": "string",
              "description": "Criticality of the query on the clinical trial results",
              "enum": ["Critical", "Major", "Minor"]
            },
            "inquiry": {
              "type": "string",
              "description": "Text of the inquiry to the study site"
            },
            "reason": {
              "type": "string",
              "description": "Justification for the inquiry"
            },
            "variables": {
              "type": "array",
              "description": "List of variables and their values related to the query",
              "items": {
                "type": "object",
                "properties": {
                  "variable": {
                    "type": "string",
                    "description": "Variable name"
                  },
                  "value": {
                    "type": "string",
                    "description": "Value of the variable"
                  }
                },
                "required": [
                  "variable",
                  "value"
                ],
                "additionalProperties": false
              }
            }
          },
          "required": [
            "query_no",
            "criticality",
            "inquiry",
            "reason",
            "variables"
          ],
          "additionalProperties": false
        }
      }
    },
    "required": [
      "usubjid",
      "timeline",
      "data_issues",
      "deviations",
      "queries"
    ],
    "additionalProperties": false
  }
}

''

SyntaxError: incomplete input (<ipython-input-25-2de6c870828f>, line 2)